<a href="https://colab.research.google.com/github/yamak493/nlf/blob/claude/kind-mendel-tdveor/mp4_to_mannequin_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕺 mp4 → モーション抽出 → 踊るマネキン動画（音声付き）

このノートブックは、**指定 URL からダウンロードした mp4 の「始点秒〜終点秒」の区間から人物の 3D モーションを抽出し、
そのモーションで動くマネキンの動画（元動画の音声付き mp4）を書き出す**ためのものです。

使用する技術:

| 役割 | 使うもの |
|---|---|
| 画像 → 3D 人体（頂点・関節・SMPL パラメータ） | [NLF (Neural Localizer Fields)](https://github.com/isarandi/nlf) の TorchScript モデル `v0.3.2` |
| 3D 頂点 → SMPL パラメータへのフィッティング | [SMPLFitter](https://github.com/isarandi/smplfitter)（NLF の内部でも使われています） |
| モーションの震え（ジッター）除去 | [MMPose](https://github.com/open-mmlab/mmpose/tree/0.x) の Smoother / 時間フィルタ（Savitzky-Golay・[SmoothNet](https://github.com/cure-lab/SmoothNet) など） |
| 動画の切り出し・音声の合成 | ffmpeg（`imageio-ffmpeg` に同梱のバイナリを使うので別途インストール不要） |
| SMPL → MMD 用 VMD モーションへの変換 | このリポジトリの [`nlf2vmd`](nlf2vmd/README.md)（仕様: [`vmd.md`](vmd.md)） |

## 処理の流れ

1. ライブラリを自動インストール（`smplfitter` など）
2. NLF の学習済みモデル（約 470&nbsp;MB）を自動ダウンロード
3. 入力動画（mp4）を URL からダウンロード（既定: `https://made-by-free.com/night-fire.mp4`）
4. 始点秒・終点秒を指定して、その区間を切り出し（映像と音声）
5. 1 フレームずつ NLF で推論 → 人物を 1 人選んで追跡 → **モーションデータ**（SMPL の pose / betas / trans / 関節 / 頂点）を取得
6. 検出できなかったフレームを埋めて `motion.npz` に保存
7. **MMPose の Smoother**（時間フィルタ）で体の震え（ジッター）を抑える
8. モーションを反映した**マネキンのメッシュ**を作ってレンダリング
9. 元動画の音声を合成して `mannequin_with_audio.mp4` を出力・再生・ダウンロード
10. モーションを **FBX**（`motion.fbx`：スケルトン＋スキン付きマネキン＋アニメーション）でも出力・ダウンロード
11. モーションを **MMD 用の VMD**（`motion.vmd`：センター・グルーブ・足ＩＫ・上半身の回転）でも出力し、足の滑り・埋まり・震えなどの評価指標とグラフを表示

## 実行前の注意

* **GPU ランタイムが必須**です。Colab では「ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: GPU」を選んでください（NLF は半精度で動くため CPU では実行できません）。
* 処理時間の目安（Colab T4 / 720p）: **推論 約 0.3〜0.6 秒/フレーム**。まずは **5〜10 秒程度**の区間で試してください。
* マネキンの見た目は 2 種類あります。
  * **パーツ凸包マネキン（既定）**: SMPL の公式ファイルが無くても動きます。木製デッサン人形のような見た目になります。
  * **SMPL メッシュ**: SMPL 公式配布ファイル（要ユーザー登録）がある場合のみ。人体そのままの滑らかなメッシュになります。
* ライセンス: NLF のモデルは**非商用の研究用途**で公開されています。SMPL 系ボディモデルは [smpl.is.tue.mpg.de](https://smpl.is.tue.mpg.de/) 等での登録・ライセンス同意が必要です。入力する動画は自分に権利があるものを使ってください。

---
## 1. ライブラリのインストール

必要なパッケージを自動で入れます（Colab では PyTorch は既に入っているのでそのまま使います）。
初回は 1〜2 分ほどかかります。

In [ ]:
#@title 1. セットアップ（実行するだけ） { display-mode: "form" }
import importlib.util
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
print('Colab 上で実行中:', IN_COLAB)


def pip_install(*pkgs):
    print('インストール中:', ' '.join(pkgs))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)


# --- PyTorch（Colab には最初から入っています） ---
try:
    import torch
    import torchvision  # NLF の TorchScript を読むために必須
except ImportError:
    pip_install('torch', 'torchvision')
    import torch
    import torchvision

# --- その他の依存パッケージ ---
required = [
    ('smplfitter', 'smplfitter'),   # SMPL フィッティング（NLF 内部でも使用）
    ('scipy', 'scipy'),
    ('trimesh', 'trimesh'),
    ('imageio', 'imageio'),
    ('imageio_ffmpeg', 'imageio-ffmpeg'),  # ffmpeg バイナリ同梱
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
    ('PIL', 'pillow'),
]
missing = [pkg for mod, pkg in required if importlib.util.find_spec(mod) is None]
if missing:
    pip_install(*missing)
else:
    print('依存パッケージはすべて揃っています。')

import numpy as np

# numpy 2.x では np.infty が削除されたが、pyrender など一部ライブラリがまだ使うので補う
if not hasattr(np, 'infty'):
    np.infty = np.inf

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print()
print('PyTorch:', torch.__version__, '/ NumPy:', np.__version__)
print('デバイス:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ GPU が見つかりません。NLF は half 精度で動作するため GPU が必要です。')
    print('   Colab なら「ランタイム → ランタイムのタイプを変更 → GPU」を選んでから、')
    print('   このノートブックを最初から実行し直してください。')

WORK_DIR = os.path.abspath('nlf_mannequin')
os.makedirs(WORK_DIR, exist_ok=True)
print('作業ディレクトリ:', WORK_DIR)

---
## 2. NLF 学習済みモデルのダウンロード

[NLF v0.3.2 リリース](https://github.com/isarandi/nlf/releases/tag/v0.3.2) の
`nlf_l_multi_0.3.2.torchscript`（約 470&nbsp;MB / EfficientNetV2-L バックボーン）を取得します。

このモデルは **人物検出 → 3D 頂点・関節の推定 → SMPL パラメータへのフィッティング（SMPLFitter）** までを
1 つの TorchScript にまとめたものです。一度ダウンロードすれば以降のセル実行では再利用されます。

In [ ]:
#@title 2. NLF モデルの自動ダウンロードと読み込み { display-mode: "form" }
import time
import urllib.request

MODEL_URL = 'https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript'
MODEL_PATH = os.path.join(WORK_DIR, 'nlf_l_multi_0.3.2.torchscript')
MIN_EXPECTED_BYTES = 300 * 1024 ** 2


def download(url, path, min_bytes=0):
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        print(f'既にダウンロード済み: {path} ({os.path.getsize(path) / 1024 ** 2:.0f} MB)')
        return path
    tmp = path + '.part'
    print('ダウンロード中:', url)
    t0 = time.time()
    with urllib.request.urlopen(url) as resp, open(tmp, 'wb') as f:
        total = int(resp.headers.get('Content-Length', 0))
        done = 0
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if total:
                bar = '█' * int(30 * done / total)
                print(f'\r  [{bar:<30}] {done / 1024 ** 2:7.0f} / {total / 1024 ** 2:.0f} MB',
                      end='', flush=True)
    print()
    os.replace(tmp, path)
    print(f'完了（{time.time() - t0:.0f} 秒）:', path)
    return path


download(MODEL_URL, MODEL_PATH, MIN_EXPECTED_BYTES)

print('モデルを読み込み中…（30 秒ほどかかります）')
nlf_model = torch.jit.load(MODEL_PATH).to(DEVICE).eval()
print('読み込み完了 ✅')

---
## 3.（任意）SMPL 公式ボディモデルの用意

* **何もしなくても動きます。** その場合は SMPL の頂点を体のパーツごとに凸包（convex hull）で包んだ
  「デッサン人形風マネキン」でレンダリングします（メッシュの面情報が不要な方式です）。
* SMPL の公式ファイルがあると、**人体メッシュそのもの**をマネキンとして描画でき、さらに
  SMPLFitter で「全フレーム共通の体型（betas）」に整えるリフィットも実行できます。

公式ファイルは [smpl.is.tue.mpg.de](https://smpl.is.tue.mpg.de/) でのユーザー登録とライセンス同意が必要です。
登録済みなら、下の `DOWNLOAD_SMPL_MODEL` を `True` にして実行すると、`smplfitter` のダウンローダ経由で
メールアドレスとパスワードを聞かれ、自動で配置されます（入力内容はどこにも保存されません）。
既に手元にファイルがある場合は `body_models/smpl/` 以下に置いてください。

In [ ]:
#@title 3. ボディモデルの検出（任意ダウンロード） { display-mode: "form" }
DOWNLOAD_SMPL_MODEL = False  #@param {type:"boolean"}

BODY_MODELS_DIR = os.path.join(WORK_DIR, 'body_models')
os.makedirs(BODY_MODELS_DIR, exist_ok=True)
os.environ['SMPLFITTER_BODY_MODELS'] = BODY_MODELS_DIR
# 想定する配置: body_models/smpl/basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl

if DOWNLOAD_SMPL_MODEL:
    import getpass
    from pathlib import Path
    from urllib.parse import quote
    try:
        from smplfitter.download import _download_smpl, _make_opener
        email = input('MPI (smpl.is.tue.mpg.de) の登録メールアドレス: ')
        password = getpass.getpass('パスワード: ')
        auth = f'username={quote(email, safe="")}&password={quote(password, safe="")}'.encode()
        _download_smpl(_make_opener(), auth, Path(BODY_MODELS_DIR))
    except Exception as e:
        print('⚠️ ダウンロードに失敗しました:', repr(e))
        print('   登録が済んでいるか、メール/パスワードが正しいかを確認してください。')
        print('   （このまま進めてもパーツ凸包マネキンで動画は作れます）')


def load_body_model(model_name='smpl', gender='neutral'):
    # 公式ボディモデルが見つかればロードする。無ければ None を返す。
    try:
        from smplfitter.pt import BodyModel
        bm = BodyModel(model_name, gender, num_betas=10)
        return bm
    except Exception as e:
        print(f'SMPL 公式ファイルは見つかりませんでした（{type(e).__name__}）。')
        return None


BODY_MODEL = load_body_model('smpl')
SMPL_FACES = None if BODY_MODEL is None else np.asarray(BODY_MODEL.faces, np.int32)
if SMPL_FACES is None:
    print('→ マネキンは「パーツ凸包」方式で作ります（公式ファイル不要）。')
else:
    print(f'→ SMPL メッシュが使えます（面数 {len(SMPL_FACES)}）。')

---
## 4. 入力動画（mp4）のダウンロード

* `VIDEO_URL`（既定: `https://made-by-free.com/night-fire.mp4`）から mp4 をダウンロードして入力にします。
* ダウンロード済みのファイルがあれば再ダウンロードはしません。
* 手元のファイルを使いたい場合は `VIDEO_PATH` にパスを書いてください（`VIDEO_URL` より優先されます）。


In [ ]:
#@title 4. 動画をダウンロード { display-mode: "form" }
import shutil
import urllib.parse
import urllib.request

VIDEO_URL = 'https://made-by-free.com/night-fire.mp4'  #@param {type:"string"}
VIDEO_PATH = ''  #@param {type:"string"}

if VIDEO_PATH:
    assert os.path.exists(VIDEO_PATH), f'ファイルが見つかりません: {VIDEO_PATH}'
else:
    assert VIDEO_URL, 'VIDEO_URL か VIDEO_PATH を指定してください。'
    fname = os.path.basename(urllib.parse.urlparse(VIDEO_URL).path) or 'input.mp4'
    VIDEO_PATH = os.path.join(WORK_DIR, fname)
    if not os.path.exists(VIDEO_PATH) or os.path.getsize(VIDEO_PATH) == 0:
        print('ダウンロード中:', VIDEO_URL)
        tmp_path = VIDEO_PATH + '.part'
        req = urllib.request.Request(VIDEO_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp, open(tmp_path, 'wb') as f:
            shutil.copyfileobj(resp, f)
        os.replace(tmp_path, VIDEO_PATH)

print('入力動画:', VIDEO_PATH, f'({os.path.getsize(VIDEO_PATH) / 1024 ** 2:.1f} MB)')

---
## 5. 動画の情報を確認して、切り出す区間などを設定

ここで **始点秒数 `START_SEC`・終点秒数 `END_SEC`** を指定します。まずは 5〜10 秒程度で試すのがおすすめです。

### 切り出し・人物

| 設定 | 意味 |
|---|---|
| `START_SEC` / `END_SEC` | 切り出す区間（秒）。`END_SEC = 0` なら動画の最後まで |
| `TARGET_FPS` | 処理する fps（`0` で元動画のまま）。小さくすると速くなります |
| `MAX_HEIGHT` | 推論時に縮小する高さの上限（720 程度が速度と精度のバランス◎） |
| `PERSON_SELECT` | 最初のフレームでどの人物を主役にするか（`largest`=一番大きく写っている人 / `center`=画面中央の人） |

### 推論（GPU の使い方と精度）

| 設定 | 意味 |
|---|---|
| `BATCH_SIZE` | 一度に GPU へ送るフレーム数 |
| `NUM_AUG` | 1 人あたりのテスト時データ拡張（TTA）の枚数。**奇数のみ**（偶数だと拡張の振り方が左右非対称になります） |
| `ANTIALIAS` | クロップを作るときの超サンプリング倍率。小さく写っている人物に効きます（NLF 本体のコメントに「4 は精度が上がることがある」とあります） |
| `DETECTOR_THRESHOLD` | 人物検出のしきい値。検出が途切れるときは 0.15 程度まで下げてください |

NLF が GPU に流すのは「**フレーム数 × 人数 × `NUM_AUG`** 枚のクロップ（384×384）」です。
1 人しか写っていない動画で `BATCH_SIZE=4`・`NUM_AUG=1` だと 4 枚しか流れず、GPU がほとんど遊びます。
**GPU メモリが余っているときは、まず `BATCH_SIZE` を増やし、次に `NUM_AUG` を増やしてください。**

`NUM_AUG` は、明るさ・面内回転・拡大率・左右反転を変えた複数のクロップで推論し、
**不確実性で重み付けした幾何中央値**で統合する仕組みです（NLF 本体の機能）。
つまり「体の細かい震え」の原因であるフレームごとの推定ノイズを、後処理ではなく**発生源で**減らせます（後処理での平滑化はセル 8a の MMPose Smoother で行います）。
所要時間はおおよそ `NUM_AUG` に比例します（1 → 3 で約 2〜3 倍）。メモリが足りなければ自動でバッチを分割して再試行します。

### 出力

| 設定 | 意味 |
|---|---|
| `MANNEQUIN_STYLE` | `auto`（公式 SMPL があればメッシュ、無ければパーツ凸包）/ `parts` / `smpl_mesh` |
| `CAMERA_MODE` | `fit`=元の視点のまま人物が画面いっぱいに映るよう自動フレーミング / `original`=元動画と同じ画角 |
| `VIEW_AZIMUTH_DEG` | マネキンを縦軸まわりに回して別角度から見る（度） |
| `SIDE_BY_SIDE` | 出力の左に元動画、右にマネキンを並べる（幅が 2 倍になります）。位置をそのまま見比べたいときは `CAMERA_MODE='original'` と併用 |

In [ ]:
#@title 5. 区間・出力の設定 { display-mode: "form" }
# --- 切り出す区間と入力の解像度 ---
START_SEC = 30.0  #@param {type:"number"}
END_SEC = 60.0  #@param {type:"number"}
TARGET_FPS = 30  #@param {type:"integer"}
MAX_HEIGHT = 720  #@param {type:"integer"}
PERSON_SELECT = "largest"  #@param ["largest", "center"]

# --- 推論（GPU が余っているなら BATCH_SIZE → NUM_AUG の順に増やす） ---
BATCH_SIZE = 128  #@param {type:"integer"}
NUM_AUG = 3  #@param [1, 3, 5, 7] {type:"raw"}
ANTIALIAS = 2  #@param [1, 2, 4] {type:"raw"}
DETECTOR_THRESHOLD = 0.25  #@param {type:"number"}

# --- 出力 ---
MANNEQUIN_STYLE = "auto"  #@param ["auto", "parts", "smpl_mesh"]
CAMERA_MODE = "fit"  #@param ["fit", "original"]
VIEW_AZIMUTH_DEG = 0  #@param {type:"slider", min:-180, max:180, step:15}
SHOW_FLOOR = True  #@param {type:"boolean"}
SIDE_BY_SIDE = False  #@param {type:"boolean"}
OUT_HEIGHT = 720  #@param {type:"integer"}

# ---- ffmpeg まわりのユーティリティ（imageio-ffmpeg 同梱のバイナリを使う） ----
import imageio.v2 as imageio
import imageio_ffmpeg


def ffmpeg_exe():
    try:
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return 'ffmpeg'


def run_ffmpeg(args, check=True):
    p = subprocess.run([ffmpeg_exe(), '-hide_banner', *args],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if check and p.returncode != 0:
        raise RuntimeError('ffmpeg 失敗:\n' + p.stdout[-3000:])
    return p


def probe(path):
    with imageio.get_reader(path) as r:
        meta = r.get_meta_data()
    info = run_ffmpeg(['-i', path], check=False).stdout
    return dict(fps=float(meta.get('fps') or 30.0),
                duration=float(meta.get('duration') or 0.0),
                size=tuple(meta.get('size') or (0, 0)),
                has_audio=('Audio:' in info))


SRC_INFO = probe(VIDEO_PATH)
print('入力動画:', SRC_INFO)

duration = SRC_INFO['duration']
START_SEC = max(0.0, float(START_SEC))
END_SEC = float(END_SEC)
if END_SEC <= 0 or (duration and END_SEC > duration):
    END_SEC = duration if duration else END_SEC
assert END_SEC > START_SEC, '終点秒数は始点秒数より後にしてください。'
FPS = float(TARGET_FPS) if TARGET_FPS and TARGET_FPS > 0 else SRC_INFO['fps']
n_expected = int(round((END_SEC - START_SEC) * FPS))
print(f'切り出す区間: {START_SEC:.2f} 秒 〜 {END_SEC:.2f} 秒'
      f'（{END_SEC - START_SEC:.2f} 秒 / 約 {n_expected} フレーム @ {FPS:g} fps）')
if n_expected > 900:
    print('⚠️ フレーム数が多いので時間がかかります。まずは短い区間で試すことをおすすめします。')

---
## 6. 指定区間の切り出し（映像 + 音声）

映像は推論しやすいように `MAX_HEIGHT` 以下に縮小し、`TARGET_FPS` に揃えます。
音声は同じ区間を `m4a` として取り出し、最後にマネキン動画へ合成します（音声トラックが無い動画でもそのまま進みます）。

In [ ]:
#@title 6. 区間を切り出す { display-mode: "form" }
SEGMENT_MP4 = os.path.join(WORK_DIR, 'segment.mp4')
SEGMENT_AUDIO = os.path.join(WORK_DIR, 'segment.m4a')

vf = [f'fps={FPS}']
if MAX_HEIGHT and MAX_HEIGHT > 0:
    # 高さが MAX_HEIGHT を超えるときだけ縮小（幅は 2 の倍数に丸める）
    vf.append(f"scale='trunc(iw*min(1,{MAX_HEIGHT}/ih)/2)*2':'trunc(ih*min(1,{MAX_HEIGHT}/ih)/2)*2'")

run_ffmpeg(['-y', '-loglevel', 'error', '-ss', f'{START_SEC:.3f}', '-to', f'{END_SEC:.3f}',
            '-i', VIDEO_PATH, '-an', '-vf', ','.join(vf),
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-pix_fmt', 'yuv420p',
            SEGMENT_MP4])
SEG_INFO = probe(SEGMENT_MP4)
print('切り出した映像:', SEG_INFO)

AUDIO_PATH = None
if SRC_INFO['has_audio']:
    try:
        run_ffmpeg(['-y', '-loglevel', 'error', '-ss', f'{START_SEC:.3f}', '-to', f'{END_SEC:.3f}',
                    '-i', VIDEO_PATH, '-vn', '-c:a', 'aac', '-b:a', '192k', SEGMENT_AUDIO])
        if os.path.getsize(SEGMENT_AUDIO) > 0:
            AUDIO_PATH = SEGMENT_AUDIO
    except Exception as e:
        print('音声の取り出しに失敗しました:', repr(e))
print('音声:', AUDIO_PATH or 'なし（無音の動画を出力します）')

---
## 7. モーション抽出（NLF 推論）

各フレームを NLF に通して、人物ごとに

* `pose` … SMPL の関節回転（回転ベクトル 24×3）
* `betas` … 体型パラメータ（10 次元）
* `trans` … 全身の位置（m）
* `joints3d` / `vertices3d` … カメラ座標系の 3D 関節・頂点（mm 単位、x=右 / y=下 / z=奥）

を得ます。NLF は内部で **SMPLFitter** を使い、推定した非パラメトリックな頂点・関節に SMPL を当てはめています。

複数人が写っている場合は、**最初のフレームで選んだ人物に最も近い検出を毎フレーム追跡**して 1 人分だけを取り出します。
検出できなかったフレームは後で前後から補間します。

In [ ]:
#@title 7. NLF でモーションを抽出 { display-mode: "form" }
from tqdm.auto import tqdm

BODY_MODEL_NAME = 'smpl'  # smplx も指定できますが、このノートブックは smpl 前提です
N_VERTS, N_JOINTS = 6890, 24
N_PREVIEW = 4  # あとで重ね描画チェックに使うフレーム数

# クロップ単位のチャンクサイズ。num_aug より小さいとチャンク分割が無効になり
# 一気に全クロップを流してしまう（メモリ不足の原因）ので下限を設ける。
INTERNAL_BATCH_SIZE = max(64, int(NUM_AUG) * 4)
print(f'1 回の呼び出しで GPU に流すクロップ: 最大 {int(BATCH_SIZE) * int(NUM_AUG)} 枚 '
      f'(BATCH_SIZE={BATCH_SIZE} x NUM_AUG={NUM_AUG}, チャンク上限 {INTERNAL_BATCH_SIZE})')


def pick_first_person(boxes, image_w):
    # boxes: (n, 5) = x, y, w, h, score
    areas = boxes[:, 2] * boxes[:, 3]
    if PERSON_SELECT == 'center':
        # ある程度大きく写っている人の中で、画面中央に最も近い人を選ぶ
        big = np.flatnonzero(areas >= 0.3 * areas.max())
        cx = boxes[big, 0] + boxes[big, 2] / 2
        return int(big[np.argmin(np.abs(cx - image_w / 2))])
    return int(np.argmax(areas))


frames_pose, frames_betas, frames_trans = [], [], []
frames_joints, frames_verts, frames_box, frames_uncert = [], [], [], []
valid = []
preview = {}

reader = imageio.get_reader(SEGMENT_MP4)
prev_trans = None
batch_imgs, batch_idx = [], []
frame_count = 0
preview_at = set(np.linspace(0, max(n_expected - 1, 0), N_PREVIEW).astype(int).tolist())


def flush(batch_imgs, batch_idx):
    global prev_trans
    if not batch_imgs:
        return
    try:
        images = torch.from_numpy(np.stack(batch_imgs)).permute(0, 3, 1, 2).contiguous().to(DEVICE)
        with torch.inference_mode(), torch.device(DEVICE):
            pred = nlf_model.detect_smpl_batched(
                images, model_name=BODY_MODEL_NAME,
                detector_threshold=float(DETECTOR_THRESHOLD),
                internal_batch_size=INTERNAL_BATCH_SIZE, num_aug=int(NUM_AUG),
                antialias_factor=int(ANTIALIAS))
    except RuntimeError as e:
        # メモリ不足のときはバッチを半分に割ってやり直す
        if 'out of memory' not in str(e).lower() or len(batch_imgs) == 1:
            raise
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        half = len(batch_imgs) // 2
        flush(batch_imgs[:half], batch_idx[:half])
        flush(batch_imgs[half:], batch_idx[half:])
        return
    for k in range(len(batch_idx)):
        boxes = pred['boxes'][k].detach().float().cpu().numpy()
        n_people = len(boxes)
        if n_people == 0:
            frames_pose.append(np.zeros(72, np.float32))
            frames_betas.append(np.zeros(10, np.float32))
            frames_trans.append(np.zeros(3, np.float32))
            frames_joints.append(np.zeros((N_JOINTS, 3), np.float32))
            frames_verts.append(np.zeros((N_VERTS, 3), np.float32))
            frames_box.append(np.zeros(5, np.float32))
            frames_uncert.append(np.nan)
            valid.append(False)
            continue
        trans = pred['trans'][k].detach().float().cpu().numpy()
        if prev_trans is None:
            j = pick_first_person(boxes, images.shape[3])
        else:
            d = np.linalg.norm(trans - prev_trans[None], axis=-1)
            j = int(np.argmin(d))
            if d[j] > 1.5:  # 追跡が外れたと判断したら一番大きい人に戻す
                j = int(np.argmax(boxes[:, 2] * boxes[:, 3]))
        prev_trans = trans[j]
        frames_pose.append(pred['pose'][k][j].detach().float().reshape(-1).cpu().numpy())
        frames_betas.append(pred['betas'][k][j].detach().float().cpu().numpy())
        frames_trans.append(trans[j])
        frames_joints.append(pred['joints3d'][k][j].detach().float().cpu().numpy())
        frames_verts.append(pred['vertices3d'][k][j].detach().float().cpu().numpy())
        frames_box.append(boxes[j])
        frames_uncert.append(
            float(pred['joint_uncertainties'][k][j].detach().float().mean().cpu()))
        valid.append(True)


pbar = tqdm(total=n_expected, desc='推論中')
for frame in reader:
    frame = np.asarray(frame)[..., :3]
    if frame_count in preview_at:
        preview[frame_count] = frame.copy()
    batch_imgs.append(frame)
    batch_idx.append(frame_count)
    frame_count += 1
    if len(batch_imgs) >= max(1, int(BATCH_SIZE)):
        flush(batch_imgs, batch_idx)
        pbar.update(len(batch_idx))
        batch_imgs, batch_idx = [], []
flush(batch_imgs, batch_idx)
pbar.update(len(batch_idx))
pbar.close()
reader.close()

valid = np.array(valid, bool)
motion_raw = dict(
    pose=np.stack(frames_pose).reshape(len(valid), -1, 3),
    betas=np.stack(frames_betas),
    trans=np.stack(frames_trans),
    joints3d=np.stack(frames_joints),
    vertices3d=np.stack(frames_verts),
    boxes=np.stack(frames_box),
)
N_FRAMES = len(valid)
UNCERTAINTY = np.array(frames_uncert, np.float32)
print(f'{N_FRAMES} フレーム処理 / 人物を検出できたフレーム: {int(valid.sum())}')
assert valid.any(), '人物が 1 人も検出できませんでした。区間や検出しきい値を変えてみてください。'
print('pose:', motion_raw['pose'].shape, ' vertices3d:', motion_raw['vertices3d'].shape, '(mm)')
if valid.any():
    u = UNCERTAINTY[valid]
    print(f'関節の推定不確実性: 中央値 {np.median(u):.0f} mm / 最大 {u.max():.0f} mm '
          f'(大きいほど推定が不安定なフレーム)')

---
## 8. モーションデータの保存

NLF の推定結果を、ほぼそのまま `motion.npz` にまとめます。**このセルでは平滑化・外れ値除去・接地補正などの整形は行いません**
（フレームごとの推定ノイズによる震えや足の滑りはそのまま残ります。震えは次のセル 8a で抑えます）。

ここで行うのは、後段の描画と FBX 書き出しに必要な最低限の処理だけです。

1. 人物が検出できなかったフレームを前後から線形補間（回転は回転行列の空間で補間）
2. **体型 `betas` をシーケンス全体の中央値 1 本に固定**（骨の長さを一定にする。FBX のスケルトンもこの体型から作ります）
3. 固定した体型で**頂点と関節を再計算**（再ポーズ）
   * SMPL 公式ファイルは**不要**です。NLF の TorchScript の中に SMPL の本体（`body_models`）が
     入っているので、それをそのまま呼び出します
   * 念のため、元のパラメータで再ポーズしてセル 7 の結果と一致するか自己検証します。
     一致しなければ再ポーズはせず、セル 7 の頂点をそのまま使います
4. `motion.npz` に保存

`motion.npz` が「抽出できたモーションデータ」です。他のツールで使いたいときはこのファイルを読み込んでください。

In [ ]:
#@title 8. 欠損補間・再ポーズして motion.npz に保存 { display-mode: "form" }
from scipy.spatial.transform import Rotation

SMPL_PARENTS = np.array(
    [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19, 20, 21], np.int32)
SMPL_JOINT_NAMES = [
    'pelvis', 'left_hip', 'right_hip', 'spine1', 'left_knee', 'right_knee', 'spine2',
    'left_ankle', 'right_ankle', 'spine3', 'left_foot', 'right_foot', 'neck', 'left_collar',
    'right_collar', 'head', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hand', 'right_hand']


def fill_gaps(arr, valid):
    # 検出できなかったフレームを、前後の有効フレームから線形補間する
    arr = np.asarray(arr, np.float32)
    idx = np.arange(len(arr))
    vi = idx[valid]
    flat = arr.reshape(len(arr), -1)[valid]
    w = np.interp(idx, vi, np.arange(len(vi)))
    lo = np.floor(w).astype(int)
    hi = np.ceil(w).astype(int)
    t = (w - lo)[:, None].astype(np.float32)
    out = flat[lo] * (1 - t) + flat[hi] * t
    return out.reshape(arr.shape)


def orthonormalize(mats):
    shape = mats.shape
    u, _, vt = np.linalg.svd(mats.reshape(-1, 3, 3))
    det = np.linalg.det(u @ vt)
    u[det < 0, :, -1] *= -1
    return (u @ vt).reshape(shape).astype(np.float32)


def fill_rotation_gaps(rotvecs, valid):
    # 回転ベクトル (T, J, 3) の欠損フレームを、回転行列の空間で補間する。
    # 重要: 回転ベクトルのまま補間してはいけない。カメラ座標系では全身の向きが
    # ほぼ 180 度回転（|回転ベクトル| ≒ π）で、表現の切れ目にちょうど乗っているため、
    # 符号が反転したフレームをまたいで線形補間すると体が 1 回転してしまう。
    rv = np.asarray(rotvecs, np.float32)
    T, J = rv.shape[:2]
    mats = Rotation.from_rotvec(rv.reshape(-1, 3)).as_matrix().reshape(T, J, 3, 3)
    mats = orthonormalize(fill_gaps(mats, valid))
    return Rotation.from_matrix(mats.reshape(-1, 3, 3)).as_rotvec().reshape(T, J, 3).astype(
        np.float32)


def jitter_metric(joints):
    # 2 階差分の大きさ = 細かい震えの指標 [mm]
    if len(joints) < 3:
        return float('nan')
    d2 = joints[2:] - 2 * joints[1:-1] + joints[:-2]
    return float(np.linalg.norm(d2, axis=-1).mean())


def get_repose_fn():
    # SMPL パラメータから頂点・関節を作り直す関数を用意する
    try:
        bm = getattr(nlf_model.body_models, BODY_MODEL_NAME)

        def fn(pose, betas, trans):
            with torch.inference_mode():
                out = bm(pose_rotvecs=pose, shape_betas=betas, trans=trans)
            return out['vertices'], out['joints']

        return fn, 'NLF の TorchScript 内の SMPL（公式ファイル不要）'
    except Exception as e:
        print('TorchScript 内の体モデルを取り出せませんでした:', repr(e))

    if BODY_MODEL is not None:
        bm2 = BODY_MODEL.to(DEVICE)

        def fn(pose, betas, trans):
            with torch.inference_mode():
                out = bm2(pose_rotvecs=pose, shape_betas=betas, trans=trans)
            return out['vertices'], out['joints']

        return fn, 'smplfitter の公式 SMPL'
    return None, None


def repose(pose, betas, trans, fn, chunk=256):
    # 戻り値は mm（体モデルは m を返すので 1000 倍する）
    verts, joints = [], []
    for st in range(0, len(pose), chunk):
        sl = slice(st, st + chunk)
        p = torch.from_numpy(np.ascontiguousarray(pose[sl])).float().to(DEVICE)
        b = torch.from_numpy(np.ascontiguousarray(betas[sl])).float().to(DEVICE)
        t = torch.from_numpy(np.ascontiguousarray(trans[sl])).float().to(DEVICE)
        v, j = fn(p.reshape(len(p), -1), b, t)
        verts.append(v.float().cpu().numpy() * 1000.0)
        joints.append(j.float().cpu().numpy() * 1000.0)
    return np.concatenate(verts), np.concatenate(joints)


# ---- 1. 欠損補間（pose だけは回転行列として補間する） ----
motion = {k: (fill_rotation_gaps(v, valid) if k == 'pose' else fill_gaps(v, valid))
          for k, v in motion_raw.items()}

# ---- 2. 体型を 1 本に固定 ----
betas_const = np.median(motion_raw['betas'][valid], axis=0).astype(np.float32)
motion['betas'] = np.tile(betas_const, (N_FRAMES, 1))

# ---- 3. 再ポーズ関数の準備と自己検証 ----
repose_fn, repose_name = get_repose_fn()
if repose_fn is not None:
    check = np.flatnonzero(valid)[:8]
    try:
        v_chk, _ = repose(motion_raw['pose'][check], motion_raw['betas'][check],
                          motion_raw['trans'][check], repose_fn, chunk=8)
        err = float(np.abs(v_chk - motion_raw['vertices3d'][check]).max())
        print(f'再ポーズの自己検証: セル 7 の頂点との最大差 {err:.3f} mm（{repose_name}）')
        if not np.isfinite(err) or err > 1.0:
            print('⚠️ 一致しなかったため、再ポーズは使わずセル 7 の頂点をそのまま使います。')
            repose_fn = None
    except Exception as e:
        print('⚠️ 再ポーズを実行できませんでした:', repr(e))
        repose_fn = None

# ---- 4. 再ポーズ（できなければセル 7 の頂点をそのまま使う） ----
if repose_fn is not None:
    motion['vertices3d'], motion['joints3d'] = repose(
        motion['pose'], motion['betas'], motion['trans'], repose_fn)

print(f'震えの指標（関節の 2 階差分）: {jitter_metric(motion["joints3d"]):.2f} mm')

# ---- 5. 保存 ----
MOTION_NPZ = os.path.join(WORK_DIR, 'motion.npz')


def save_motion():
    # motion を書き換えるセル（8a / 8b）の最後でも呼ぶこと。
    # そうしないと .npz と描画結果が食い違う。
    np.savez_compressed(
        MOTION_NPZ,
        pose=motion['pose'].astype(np.float32),              # (T, 24, 3) 回転ベクトル [rad]
        betas=motion['betas'].astype(np.float32),            # (T, 10)
        betas_const=betas_const,                             # (10,) シーケンス共通の体型
        trans=motion['trans'].astype(np.float32),            # (T, 3) [m]
        joints3d=motion['joints3d'].astype(np.float32),      # (T, 24, 3) [mm] カメラ座標系
        vertices3d=motion['vertices3d'].astype(np.float32),  # (T, 6890, 3) [mm]
        valid=valid,
        joint_uncertainty=UNCERTAINTY,                       # (T,) [mm]
        smooth_filter=np.array(SMOOTH_APPLIED),              # 8a で掛けた MMPose のフィルタ（'none' = なし）
        repose_ok=np.bool_(repose_fn is not None),
        fps=np.float32(FPS),
        start_sec=np.float32(START_SEC),
        end_sec=np.float32(END_SEC),
        num_aug=np.int32(NUM_AUG),
        joint_names=np.array(SMPL_JOINT_NAMES),
        kintree_parents=SMPL_PARENTS,
        body_model=np.array(BODY_MODEL_NAME),
    )
    return MOTION_NPZ


# セル 8a はこの（平滑化前の）状態から毎回作り直す
MOTION_UNSMOOTHED = dict(motion)
SMOOTH_APPLIED = 'none'
save_motion()
print('モーションデータを保存しました:', MOTION_NPZ,
      f'({os.path.getsize(MOTION_NPZ) / 1024 ** 2:.1f} MB)')
print('共通の体型 betas:', np.round(betas_const, 3))

### 8a. MMPose の Smoother で震え（ジッター）を抑える

NLF は 1 フレームずつ独立に推定するので、そのままだと体が細かく震えます。
ここでは [MMPose](https://github.com/open-mmlab/mmpose/tree/0.x) の **Smoother（時間フィルタ）** で、この震えを抑えます。

* 平滑化するのは SMPL の**関節の回転**（回転行列にしてから平滑化し、SVD で回転行列に戻します）と**全身の位置 `trans`** です。
  そのあと体モデルで頂点と関節を作り直すので、**動画・`motion.npz`・FBX のすべてに同じ平滑化が反映**されます
* フィルタの設定値は MMPose の `configs/_base_/filters/*.py` と同じです。SmoothNet の重みも MMPose 公式のものをダウンロードします（1 つ 3 MB 程度）
* このセルは毎回**セル 8 の直後（平滑化前）の状態から**作り直すので、フィルタを変えて何度実行しても平滑化は重なりません。
  8b のリフィットを使う場合は、このセルの後に 8b を実行してください
* 実行すると「震えの指標」が平滑化の前後で表示されます。効き具合の目安にしてください

| `SMOOTH_FILTER` | 特徴 |
|---|---|
| `savizky_golay`（既定） | 11 フレームの窓に 2 次式を当てはめる Savitzky-Golay フィルタ。**キメのポーズや速い動きが削れにくい**ので、ダンスでもキレが残ります |
| `smoothnet_t8` / `smoothnet_t16` / `smoothnet_t32` / `smoothnet_t64` | 学習済みの平滑化ネットワーク [SmoothNet](https://arxiv.org/abs/2112.13715)（Human3.6M で学習）。数字は窓の長さ（フレーム数）で、大きいほど強く効きます。震えはほぼ消えますが、30 fps では **2〜3 Hz を超える速い動きも鈍ります**。震えが残るときや、ゆっくりした動き向け |
| `gaussian` | 中央値フィルタ（11 フレーム）＋ガウシアン（σ = 4 フレーム）。いちばん強く効きますが、30 fps では 1 Hz 程度の動きから鈍ります |
| `one_euro` | 1€ フィルタ。本来はリアルタイム用（過去のフレームだけを使う）で、既定値は 2D 画像のピクセル座標向けです。このノートブックの単位では**動きが遅れる**ので非推奨 |
| `none` | 平滑化しない |

* フィルタの窓はフレーム数で決まるので、`TARGET_FPS` を上げると同じ設定でも効きが弱く（時間的に短く）なります
* MMPose 1.x では Smoother が削除されており、0.x は `mmcv-full` 1.x が必要で今の Colab（PyTorch 2.x）には入りません。
  そこで MMPose 0.x の時間フィルタ（`mmpose/core/post_processing/temporal_filters`）を、計算内容はそのままこのセルに移植しています
  （mmcv への依存だけを外したもので、MMPose の元のコードと出力が一致することを確認済みです。Apache-2.0）

In [ ]:
#@title 8a. MMPose の Smoother で震えを抑える { display-mode: "form" }
SMOOTH_FILTER = "savizky_golay"  #@param ["savizky_golay", "smoothnet_t8", "smoothnet_t16", "smoothnet_t32", "smoothnet_t64", "gaussian", "one_euro", "none"]

import math

from scipy.ndimage import gaussian_filter1d
from scipy.signal import medfilt, savgol_filter
from torch import nn

# ==== MMPose 0.x の時間フィルタ（mmpose/core/post_processing/temporal_filters）の移植 ====
# Copyright (c) OpenMMLab. Apache License 2.0 - https://github.com/open-mmlab/mmpose/tree/0.x
# mmcv への依存（Registry / load_checkpoint）だけを外し、計算内容は元のコードのまま。
# 入出力はどれも (T, K, C) = (フレーム, キーポイント, 座標) の配列。


class GaussianFilter:
    # 中央値フィルタ -> ガウシアンフィルタ（human_dynamics の smooth_bbox 由来）
    def __init__(self, window_size=11, sigma=4.0):
        assert window_size % 2 == 1, f'window_size は奇数にしてください: {window_size}'
        self.window_size, self.sigma = window_size, sigma

    def __call__(self, x):
        T = x.shape[0]
        if T < self.window_size:
            x = np.pad(x, [(self.window_size - T, 0), (0, 0), (0, 0)], mode='edge')
        smoothed = medfilt(x, (self.window_size, 1, 1))
        smoothed = gaussian_filter1d(smoothed, self.sigma, axis=0)
        return smoothed[-T:]


class SavizkyGolayFilter:
    # 窓の中を多項式で当てはめる Savitzky-Golay フィルタ（ピークが削れにくい）
    def __init__(self, window_size=11, polyorder=2):
        assert 0 < polyorder < window_size, 'polyorder は 1 以上 window_size 未満にしてください'
        self.window_size, self.polyorder = window_size, polyorder

    def __call__(self, x):
        T = x.shape[0]
        if T < self.window_size:
            x = np.pad(x, [(self.window_size - T, 0), (0, 0), (0, 0)], mode='edge')
        smoothed = savgol_filter(x, self.window_size, self.polyorder, axis=0)
        return smoothed[-T:]


class OneEuroFilter:
    # 1€ フィルタ（VIBE の smooth_pose 由来）。過去のフレームだけを使うので動きが少し遅れる
    def __init__(self, min_cutoff=0.004, beta=0.7, d_cutoff=1.0):
        self.window_size = 1
        self.min_cutoff, self.beta, self.d_cutoff = float(min_cutoff), float(beta), float(d_cutoff)

    def __call__(self, x):
        def alpha(cutoff):   # フレーム間隔 t_e = 1 での平滑化係数
            r = 2 * math.pi * cutoff
            return r / (r + 1)

        out = x.copy()
        x_prev, dx_prev = x[0], 0.0
        for t in range(1, len(x)):
            a_d = alpha(self.d_cutoff)
            dx_hat = a_d * (x[t] - x_prev) + (1 - a_d) * dx_prev
            a = alpha(self.min_cutoff + self.beta * np.abs(dx_hat))
            x_prev, dx_prev = a * x[t] + (1 - a) * x_prev, dx_hat
            out[t] = x_prev
        return out


class SmoothNetResBlock(nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.5):
        super().__init__()
        self.linear1 = nn.Linear(in_channels, hidden_channels)
        self.linear2 = nn.Linear(hidden_channels, in_channels)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
        self.dropout = nn.Dropout(p=dropout, inplace=True)

    def forward(self, x):
        identity = x
        x = self.lrelu(self.dropout(self.linear1(x)))
        x = self.lrelu(self.dropout(self.linear2(x)))
        return x + identity


class SmoothNet(nn.Module):
    # 時間方向だけを見る全結合ネット。チャンネル（キーポイント x 座標）ごとに独立に処理する
    def __init__(self, window_size, output_size, hidden_size=512, res_hidden_size=256,
                 num_blocks=3, dropout=0.5):
        super().__init__()
        assert output_size <= window_size
        self.window_size, self.output_size, self.hidden_size = window_size, output_size, hidden_size
        self.encoder = nn.Sequential(nn.Linear(window_size, hidden_size),
                                     nn.LeakyReLU(0.1, inplace=True))
        self.res_blocks = nn.Sequential(*[
            SmoothNetResBlock(hidden_size, res_hidden_size, dropout) for _ in range(num_blocks)])
        self.decoder = nn.Linear(hidden_size, output_size)

    def forward(self, x):   # (N, C, T) -> (N, C, T)
        N, C, T = x.shape
        num_windows = T - self.window_size + 1
        x = x.unfold(2, self.window_size, 1)                  # 1 フレームずつずらした窓
        x = self.decoder(self.res_blocks(self.encoder(x)))    # (N, C, 窓の数, output_size)
        out, count = x.new_zeros(N, C, T), x.new_zeros(T)
        for t in range(num_windows):                          # 重なった窓の出力を平均する
            out[..., t:t + self.output_size] += x[:, :, t]
            count[t:t + self.output_size] += 1.0
        return out.div(count)


class SmoothNetFilter:
    def __init__(self, window_size, output_size, checkpoint=None, hidden_size=512,
                 res_hidden_size=256, num_blocks=3, device='cpu', root_index=None):
        self.window_size, self.device, self.root_index = window_size, device, root_index
        self.smoothnet = SmoothNet(window_size, output_size, hidden_size, res_hidden_size,
                                   num_blocks)
        if checkpoint:
            path = download(checkpoint, os.path.join(WORK_DIR, os.path.basename(checkpoint)))
            state = torch.load(path, map_location='cpu', weights_only=True)
            self.smoothnet.load_state_dict(state.get('state_dict', state))
        self.smoothnet.to(device).eval().requires_grad_(False)

    def __call__(self, x):
        root_index = self.root_index
        if root_index is not None:
            x_root = x[:, root_index:root_index + 1]
            x = np.delete(x, root_index, axis=1) - x_root
        T, K, C = x.shape
        if T < self.window_size:
            smoothed = x   # 窓より短い系列は平滑化しない
        else:
            # MMPose は全チャンネルを一度に流すが、頂点のようにチャンネルが多いと GPU メモリが
            # 足りないので、中間特徴が 2^26 要素（256 MB）程度に収まるよう分割する
            # （チャンネルごとに独立な計算なので結果は同じ）
            chunk = max(1, 2 ** 26 // ((T - self.window_size + 1) * self.smoothnet.hidden_size))
            xt = torch.tensor(x.reshape(T, K * C).T, dtype=torch.float32)   # (K*C, T)
            with torch.no_grad():
                out = [self.smoothnet(xt[None, s:s + chunk].to(self.device))[0].cpu()
                       for s in range(0, K * C, chunk)]
            smoothed = torch.cat(out).T.numpy().reshape(T, K, C).astype(x.dtype)
        if root_index is not None:
            smoothed = np.concatenate(
                (smoothed[:, :root_index] + x_root, x_root, smoothed[:, root_index:] + x_root),
                axis=1)
        return smoothed


MMPOSE_FILTERS = dict(GaussianFilter=GaussianFilter, SavizkyGolayFilter=SavizkyGolayFilter,
                      OneEuroFilter=OneEuroFilter, SmoothNetFilter=SmoothNetFilter)


def build_filter(cfg):
    cfg = dict(cfg)
    return MMPOSE_FILTERS[cfg.pop('type')](**cfg)


# MMPose の configs/_base_/filters/*.py と同じ設定
_SMOOTHNET_URL = 'https://download.openmmlab.com/mmpose/plugin/smoothnet/smoothnet_ws{}_h36m.pth'
MMPOSE_FILTER_CFGS = {
    'one_euro': dict(type='OneEuroFilter', min_cutoff=0.004, beta=0.7),
    'gaussian': dict(type='GaussianFilter', window_size=11, sigma=4.0),
    'savizky_golay': dict(type='SavizkyGolayFilter', window_size=11, polyorder=2),
    **{f'smoothnet_t{ws}': dict(type='SmoothNetFilter', window_size=ws, output_size=ws,
                                checkpoint=_SMOOTHNET_URL.format(ws), hidden_size=512,
                                res_hidden_size=256, num_blocks=3, root_index=0)
       for ws in (8, 16, 32, 64)},
}


def mmpose_smooth(x, filter_cfg):
    # Smoother.smooth()（オフライン・1 人分）と同じく、系列 (T, K, C) 全体にフィルタを 1 回掛ける。
    # MMPose に無い処理として、系列の平均を引いてから掛けて戻す。SmoothNet は値の位置で
    # 結果が変わるため（奥行き 3〜10 m の trans をそのまま入れると崩れる）。
    # 他のフィルタは平行移動しても結果がほぼ変わらない。
    x = np.asarray(x, np.float32)
    mean = x.mean(0, keepdims=True)
    return build_filter(filter_cfg)(x - mean) + mean


# ---- 平滑化（毎回セル 8 の直後の状態から作り直すので、何度実行しても重ならない） ----
motion = dict(MOTION_UNSMOOTHED)
SMOOTH_APPLIED = 'none'
if SMOOTH_FILTER != 'none':
    cfg = dict(MMPOSE_FILTER_CFGS[SMOOTH_FILTER])
    if cfg['type'] == 'SmoothNetFilter':
        # root_index=0 は 3D キーポイント用（骨盤を原点にしてから掛ける）。
        # 回転行列や全身の位置の系列には当てはまらないので外す
        cfg.update(root_index=None, device=DEVICE)
    print('MMPose のフィルタ:', cfg)
    T = len(motion['pose'])
    # 関節の回転は回転行列 (T, 24, 9) の空間で平滑化し、SVD で回転行列に戻す
    # （回転ベクトルのままだと ±π の切れ目で値が飛ぶ。fill_rotation_gaps と同じ理由）
    mats = Rotation.from_rotvec(motion['pose'].reshape(-1, 3)).as_matrix().reshape(T, -1, 9)
    mats = orthonormalize(mmpose_smooth(mats, cfg).reshape(-1, 3, 3))
    motion['pose'] = Rotation.from_matrix(mats).as_rotvec().reshape(T, -1, 3).astype(np.float32)
    motion['trans'] = mmpose_smooth(motion['trans'][:, None], cfg)[:, 0]
    if repose_fn is not None:
        motion['vertices3d'], motion['joints3d'] = repose(
            motion['pose'], motion['betas'], motion['trans'], repose_fn)
    else:
        # 再ポーズできないときは、MMPose 本来の使い方どおり 3D キーポイント（関節＋頂点）を直接平滑化する
        kp = np.concatenate([motion['joints3d'], motion['vertices3d']], axis=1) / 1000.0
        kp = mmpose_smooth(kp, cfg) * 1000.0
        motion['joints3d'], motion['vertices3d'] = kp[:, :N_JOINTS], kp[:, N_JOINTS:]
    SMOOTH_APPLIED = SMOOTH_FILTER

print(f'震えの指標（関節の 2 階差分）: 平滑化前 {jitter_metric(MOTION_UNSMOOTHED["joints3d"]):.2f} mm'
      f' -> 平滑化後 {jitter_metric(motion["joints3d"]):.2f} mm')
save_motion()
print('motion.npz を更新しました:', MOTION_NPZ, f'（フィルタ: {SMOOTH_APPLIED}）')

### 8b.（任意）SMPLFitter で全フレーム共通の体型に整える

SMPL 公式ファイルがある場合のみの**代替手段**です（既定はオフ）。
セル 8 で体型はすでに中央値 1 本に固定しているので通常は不要ですが、
[SMPLFitter](https://github.com/isarandi/smplfitter) の `share_beta=True` を使うと、
体型を共有したまま**頂点から SMPL パラメータを当てはめ直す**ことができます。
公式ファイルを持っていて、体型の推定をやり直したい場合に有効化してください。

In [ ]:
#@title 8b. 体型を共通化するリフィット（公式 SMPL がある場合のみ） { display-mode: "form" }
REFIT_SHARED_SHAPE = False  #@param {type:"boolean"}

if REFIT_SHARED_SHAPE and BODY_MODEL is not None:
    from smplfitter.pt import BodyFitter
    bm = BODY_MODEL.to(DEVICE)
    fitter = BodyFitter(bm).to(DEVICE)
    verts_t = torch.from_numpy(motion['vertices3d'] / 1000.0).float().to(DEVICE)
    joints_t = torch.from_numpy(motion['joints3d'] / 1000.0).float().to(DEVICE)
    fits = []
    # 体型はバッチ内で共有されるため、可能な限り 1 回で全フレームを流す
    # （チャンクに分けると境界ごとに体型が変わり、周期的な段差が出る）
    chunk = len(verts_t) if len(verts_t) <= 512 else 64
    with torch.inference_mode():
        for s in range(0, len(verts_t), chunk):
            fits.append(fitter.fit(target_vertices=verts_t[s:s + chunk],
                                   target_joints=joints_t[s:s + chunk],
                                   num_iter=3, beta_regularizer=1.0, share_beta=True,
                                   final_adjust_rots=True,
                                   requested_keys=['pose_rotvecs', 'shape_betas', 'trans']))
        pose_rotvecs = torch.cat([f['pose_rotvecs'] for f in fits])
        shape_betas = torch.cat([f['shape_betas'] for f in fits])
        trans = torch.cat([f['trans'] for f in fits])
        shape_betas = shape_betas.mean(0, keepdim=True).expand_as(shape_betas).contiguous()
        out = bm(pose_rotvecs=pose_rotvecs, shape_betas=shape_betas, trans=trans)
    motion['pose'] = pose_rotvecs.reshape(len(verts_t), -1, 3).cpu().numpy()
    motion['betas'] = shape_betas.cpu().numpy()
    motion['trans'] = trans.cpu().numpy()
    motion['vertices3d'] = out['vertices'].cpu().numpy() * 1000.0
    motion['joints3d'] = out['joints'].cpu().numpy() * 1000.0
    save_motion()
    print('共通体型でリフィットしました。betas =', np.round(motion['betas'][0], 3))
else:
    print('スキップしました（公式 SMPL ファイルが無い、または REFIT_SHARED_SHAPE=False）。')

---
## 9. 抽出結果の確認（元フレームへの重ね描画）

推定した 3D 頂点を元のフレームに投影して重ねます。人物にきれいに重なっていれば抽出は成功です。
ずれている場合は、区間を変える・`PERSON_SELECT` を変える・`MAX_HEIGHT` を上げる、などを試してください。

In [ ]:
#@title 9. 重ね描画でチェック { display-mode: "form" }
import matplotlib.pyplot as plt


def intrinsics_from_fov(fov_degrees, imshape):
    # NLF が既定で仮定しているカメラ（対角ではなく長辺基準の画角 55 度）と同じ式
    h, w = imshape[:2]
    f = float(max(h, w)) / (2.0 * np.tan(np.deg2rad(fov_degrees) / 2.0))
    return np.array([[f, 0, w / 2.0], [0, f, h / 2.0], [0, 0, 1]], np.float32)


def project(points_cam, K):
    z = np.maximum(points_cam[..., 2:], 1e-3)
    uv = points_cam[..., :2] / z
    return uv * np.array([K[0, 0], K[1, 1]], np.float32) + np.array([K[0, 2], K[1, 2]], np.float32)


SEG_W, SEG_H = SEG_INFO['size']
K_ORIG = intrinsics_from_fov(55.0, (SEG_H, SEG_W))

cam_verts = motion['vertices3d']

keys = sorted(k for k in preview if k < N_FRAMES)
if keys:
    fig, axes = plt.subplots(1, len(keys), figsize=(4 * len(keys), 4 * SEG_H / max(SEG_W, 1)))
    axes = np.atleast_1d(axes)
    for ax, k in zip(axes, keys):
        uv = project(cam_verts[k], K_ORIG)
        ax.imshow(preview[k])
        ax.scatter(uv[::12, 0], uv[::12, 1], s=1.0, c='lime', alpha=0.45)
        ax.set_title(f'frame {k}' + ('' if valid[k] else ' (interpolated)'), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('プレビュー用のフレームがありません。')

---
## 10. マネキンのメッシュを組み立てる

**方式 A: SMPL メッシュ**（公式ファイルがある場合）
: SMPL の面情報（13,776 三角形）をそのまま使い、推定された頂点をフレームごとに差し替えます。

**方式 B: パーツ凸包マネキン**（既定 / 公式ファイル不要）
: SMPL の 6,890 頂点を体のパーツ（関節）ごとに分け、**パーツごとの凸包**を取って
  デッサン人形のような多面体マネキンを作ります。
  頂点の所属パーツは、TorchScript モデルの中に入っているスキニングウェイト（LBS weights）から取得し、
  取れない場合は「最も近い関節」で代用します。
  凸包の面の構成（トポロジ）は 1 フレームだけで計算し、以降のフレームは同じ面構成のまま頂点を差し替えるので高速です。

In [ ]:
#@title 10. マネキンのメッシュを作る { display-mode: "form" }
from scipy.spatial import ConvexHull


def try_get_lbs_weights(model, model_name, n_verts):
    # NLF の TorchScript には SMPL のバッファが入っているので、そこからスキニングウェイトを拝借する
    try:
        bm = getattr(model.body_models, model_name)
        w = bm.weights.detach().float().cpu().numpy()
        if w.ndim == 2 and w.shape[0] == n_verts:
            return w
    except Exception:
        pass
    return None


def vertex_part_labels(verts_ref, joints_ref, lbs_weights=None):
    if lbs_weights is not None:
        return lbs_weights.argmax(-1).astype(np.int32)
    d = np.linalg.norm(verts_ref[:, None, :] - joints_ref[None, :, :], axis=-1)
    return d.argmin(1).astype(np.int32)


def build_part_hull_topology(verts_ref, labels, num_parts, min_points=8):
    # パーツごとの凸包 -> 「頂点インデックス列 + 固定の面」に変換する
    idx_chunks, face_chunks, face_part = [], [], []
    offset = 0
    for p in range(num_parts):
        member = np.flatnonzero(labels == p)
        if len(member) < min_points:
            continue
        pts = verts_ref[member]
        try:
            hull = ConvexHull(pts, qhull_options='QJ')
        except Exception:
            continue
        used = np.unique(hull.simplices)
        remap = np.full(len(member), -1, np.int64)
        remap[used] = np.arange(len(used))
        tris = remap[hull.simplices]
        v = pts[used]
        a, b, c = v[tris[:, 0]], v[tris[:, 1]], v[tris[:, 2]]
        n = np.cross(b - a, c - a)
        flip = np.einsum('ij,ij->i', n, (a + b + c) / 3.0 - v.mean(0)) < 0
        tris[flip] = tris[flip][:, ::-1]
        idx_chunks.append(member[used])
        face_chunks.append(tris + offset)
        face_part.append(np.full(len(tris), p, np.int32))
        offset += len(used)
    return (np.concatenate(idx_chunks), np.concatenate(face_chunks).astype(np.int32),
            np.concatenate(face_part))


VERTS = motion['vertices3d']                 # (T, V, 3) [mm]
JOINTS = motion['joints3d']                  # (T, J, 3) [mm]
ref = int(np.flatnonzero(valid)[len(np.flatnonzero(valid)) // 2])  # 代表フレーム

use_smpl_mesh = (MANNEQUIN_STYLE == 'smpl_mesh'
                 or (MANNEQUIN_STYLE == 'auto' and SMPL_FACES is not None))
if use_smpl_mesh and SMPL_FACES is None:
    print('⚠️ SMPL 公式ファイルが無いのでパーツ凸包マネキンに切り替えます。')
    use_smpl_mesh = False

if use_smpl_mesh:
    VERTEX_MAP = np.arange(VERTS.shape[1])
    FACES = SMPL_FACES
    print(f'SMPL メッシュを使用します: 頂点 {len(VERTEX_MAP)} / 面 {len(FACES)}')
else:
    lbs = try_get_lbs_weights(nlf_model, BODY_MODEL_NAME, VERTS.shape[1])
    print('スキニングウェイト:', 'TorchScript から取得' if lbs is not None else '最近傍関節で代用')
    LABELS = vertex_part_labels(VERTS[ref], JOINTS[ref], lbs)
    VERTEX_MAP, FACES, FACE_PART = build_part_hull_topology(
        VERTS[ref], LABELS, num_parts=JOINTS.shape[1])
    print(f'パーツ凸包マネキン: パーツ {len(np.unique(FACE_PART))} / '
          f'頂点 {len(VERTEX_MAP)} / 面 {len(FACES)}')

MESH_VERTS = VERTS[:, VERTEX_MAP] / 1000.0   # (T, M, 3) [m]
print('マネキンの頂点列:', MESH_VERTS.shape)

---
## 11. レンダリング

カメラ座標系（x=右 / y=下 / z=奥）のまま描画します。

* `CAMERA_MODE='fit'` … 元動画と同じ視点のまま、シーケンス全体が画面に収まるよう焦点距離と中心を自動調整
* `CAMERA_MODE='original'` … 元動画とまったく同じ画角（人物の位置もそのまま）
* `VIEW_AZIMUTH_DEG` … 縦軸まわりに回転させて別アングルから撮影
* `SHOW_FLOOR` … 足元の最下点に市松模様の床を敷きます（奥行きが分かりやすくなります）

レンダラは OpenGL 不要の**ソフトウェアレンダラ**（三角形を奥から順に塗る画家アルゴリズム）です。
Colab でも追加インストール無しで確実に動きます。`pyrender` が使える環境なら `RENDER_BACKEND='pyrender'`
にすると、より陰影のきれいな描画になります（失敗したら自動でソフトウェアに戻ります）。

In [ ]:
#@title 11. マネキン動画をレンダリング { display-mode: "form" }
RENDER_BACKEND = "software"  #@param ["software", "pyrender"]
BODY_COLOR = "#D8D2C6"  #@param {type:"string"}
BG_COLOR = "#1C2029"  #@param {type:"string"}

from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.collections import PolyCollection
from matplotlib.figure import Figure


def hex2rgb(s):
    s = s.lstrip('#')
    return np.array([int(s[i:i + 2], 16) / 255.0 for i in (0, 2, 4)], np.float32)


def rotate_about_y(points, center, degrees):
    th = np.deg2rad(degrees)
    c, s = np.cos(th), np.sin(th)
    R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], np.float32)  # y 軸（上下）まわり
    return (points - center) @ R.T + center


def fit_camera(points, imshape, margin=0.14):
    h, w = imshape[:2]
    p = points.reshape(-1, 3)
    p = p[p[:, 2] > 1e-3]
    u, v = p[:, 0] / p[:, 2], p[:, 1] / p[:, 2]
    u0, u1 = np.percentile(u, 0.1), np.percentile(u, 99.9)
    v0, v1 = np.percentile(v, 0.1), np.percentile(v, 99.9)
    f = min(w / max(u1 - u0, 1e-6), h / max(v1 - v0, 1e-6)) * (1.0 - margin)
    return np.array([[f, 0, w / 2 - f * (u0 + u1) / 2],
                     [0, f, h / 2 - f * (v0 + v1) / 2], [0, 0, 1]], np.float32)


def make_floor_grid(center_xz, y_level, half_size, n_cells=14):
    xs = np.linspace(center_xz[0] - half_size, center_xz[0] + half_size, n_cells + 1)
    zs = np.linspace(center_xz[1] - half_size, center_xz[1] + half_size, n_cells + 1)
    verts, faces, shade = [], [], []
    for i in range(n_cells):
        for j in range(n_cells):
            o = len(verts)
            verts += [[xs[i], y_level, zs[j]], [xs[i + 1], y_level, zs[j]],
                      [xs[i + 1], y_level, zs[j + 1]], [xs[i], y_level, zs[j + 1]]]
            faces += [[o, o + 1, o + 2], [o, o + 2, o + 3]]
            c = 0.80 if (i + j) % 2 == 0 else 0.66
            shade += [c, c]
    return (np.array(verts, np.float32), np.array(faces, np.int32),
            np.array(shade, np.float32)[:, None] * np.ones(3, np.float32))


def shade_faces(verts, faces, base_color, light_dir=(0.35, -0.75, -0.55),
                ambient=0.42, diffuse=0.72):
    a, b, c = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
    n = np.cross(b - a, c - a)
    n = n / np.maximum(np.linalg.norm(n, axis=-1, keepdims=True), 1e-9)
    l = np.asarray(light_dir, np.float32)
    l = l / np.linalg.norm(l)
    lam = np.abs(n @ l)
    return np.clip(np.asarray(base_color, np.float32)[None] * (ambient + diffuse * lam)[:, None],
                   0, 1)


def render_software(verts, faces, K, imshape, body_color, bg_color, extra=None):
    h, w = imshape[:2]
    V, F, C = [verts], [faces], [shade_faces(verts, faces, body_color)]
    n_body_faces = len(faces)
    if extra is not None:
        ev, ef, ec = extra
        V.append(ev)
        F.append(ef + len(verts))
        C.append(np.clip(shade_faces(ev, ef, (1.0, 1.0, 1.0)) * ec, 0, 1))
    V, F, C = np.concatenate(V), np.concatenate(F), np.concatenate(C)
    tri = V[F]
    depth = tri[..., 2].mean(1)
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    facing = np.einsum('ij,ij->i', n, tri.mean(1)) < 0   # カメラ（原点）を向いている面だけ描く
    facing[n_body_faces:] = True                        # 床は両面描画
    keep = (depth > 1e-3) & facing
    F, C, depth = F[keep], C[keep], depth[keep]
    order = np.argsort(-depth)                          # 奥から手前へ
    polys = project(V, K)[F[order]]

    fig = Figure(figsize=(w / 100.0, h / 100.0), dpi=100)
    canvas = FigureCanvasAgg(fig)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.axis('off')
    fig.patch.set_facecolor(bg_color)
    ax.set_facecolor(bg_color)
    ax.add_collection(PolyCollection(polys, facecolors=C[order], edgecolors='none'))
    canvas.draw()
    return np.asarray(canvas.buffer_rgba())[..., :3].copy()


class PyrenderBackend:
    def __init__(self, imshape, bg_color):
        os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
        import pyrender
        import trimesh
        self.pyrender, self.trimesh = pyrender, trimesh
        self.renderer = pyrender.OffscreenRenderer(imshape[1], imshape[0])
        self.bg = bg_color

    def render(self, verts, faces, K, imshape, body_color, bg_color, extra=None):
        pyrender, trimesh = self.pyrender, self.trimesh
        scene = pyrender.Scene(bg_color=[*bg_color, 1.0], ambient_light=(0.45, 0.45, 0.45))
        flip = np.array([1, -1, -1], np.float32)   # OpenCV 座標系 -> OpenGL 座標系
        mat = pyrender.MetallicRoughnessMaterial(
            metallicFactor=0.15, roughnessFactor=0.7, alphaMode='OPAQUE',
            baseColorFactor=[*body_color, 1.0], doubleSided=True)
        scene.add(pyrender.Mesh.from_trimesh(trimesh.Trimesh(verts * flip, faces), material=mat))
        if extra is not None:
            ev, ef, ec = extra
            m = trimesh.Trimesh(ev * flip, ef, process=False)
            m.visual.face_colors = np.concatenate(
                [np.clip(ec, 0, 1), np.ones((len(ec), 1), np.float32)], axis=1)
            scene.add(pyrender.Mesh.from_trimesh(m, smooth=False))
        cam = pyrender.IntrinsicsCamera(fx=float(K[0, 0]), fy=float(K[1, 1]),
                                        cx=float(K[0, 2]), cy=float(K[1, 2]),
                                        znear=0.05, zfar=100.0)
        scene.add(cam, pose=np.eye(4))
        for pose in _raymond_light_poses():
            scene.add(pyrender.DirectionalLight(color=np.ones(3), intensity=2.0), pose=pose)
        color, _ = self.renderer.render(scene)
        return np.asarray(color)[..., :3]


def _raymond_light_poses():
    poses = []
    for phi in [0.0, 2 * np.pi / 3, 4 * np.pi / 3]:
        theta = np.pi / 6
        z = np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])
        z /= np.linalg.norm(z)
        x = np.array([-z[1], z[0], 0.0])
        x = x / np.linalg.norm(x) if np.linalg.norm(x) > 0 else np.array([1.0, 0.0, 0.0])
        m = np.eye(4)
        m[:3, :3] = np.c_[x, np.cross(z, x), z]
        poses.append(m)
    return poses


def fulcrum_marker_geometry(points, weights, size=0.05, thresh=0.15):
    on = np.flatnonzero(weights > thresh)
    if len(on) == 0:
        return None
    V, F, C = [], [], []
    for k in on:
        w = float(np.clip(weights[k], 0.0, 1.0))
        V.append(OCTA_V * (size * (0.6 + 0.5 * w)) + points[k])
        F.append(OCTA_F + len(V[-1]) * (len(V) - 1))
        C.append(np.tile(MARKER_OFF * (1 - w) + MARKER_ON * w, (len(OCTA_F), 1)))
    return np.concatenate(V), np.concatenate(F), np.concatenate(C)


# ---- 出力サイズ・カメラ・床の準備 ----
out_h = int(OUT_HEIGHT) // 2 * 2
out_w = int(round(out_h * SEG_W / max(SEG_H, 1))) // 2 * 2
IMSHAPE = (out_h, out_w)

render_verts = MESH_VERTS.copy()
if VIEW_AZIMUTH_DEG:
    center = np.median(render_verts.reshape(-1, 3), axis=0)
    render_verts = rotate_about_y(render_verts, center, VIEW_AZIMUTH_DEG)

if CAMERA_MODE == 'original' and not VIEW_AZIMUTH_DEG:
    K_RENDER = K_ORIG * np.array([[out_w / SEG_W], [out_h / SEG_H], [1.0]], np.float32)
else:
    K_RENDER = fit_camera(render_verts[::max(1, len(render_verts) // 60)], IMSHAPE)

floor = None
if SHOW_FLOOR:
    y_floor = float(np.percentile(render_verts[..., 1], 99.7))
    xz = render_verts.reshape(-1, 3)[:, [0, 2]]
    span = float(max(np.ptp(np.percentile(xz[:, 0], [1, 99])),
                     np.ptp(np.percentile(xz[:, 1], [1, 99]))))
    floor = make_floor_grid([float(np.median(xz[:, 0])), float(np.median(xz[:, 1]))],
                            y_floor, half_size=max(1.2, span * 1.5 + 0.8))

body_rgb = hex2rgb(BODY_COLOR)
bg_rgb = hex2rgb(BG_COLOR)
backend = None
if RENDER_BACKEND == 'pyrender':
    try:
        backend = PyrenderBackend(IMSHAPE, bg_rgb)
        print('pyrender を使います。')
    except Exception as e:
        print('pyrender を初期化できませんでした（', repr(e), '）→ ソフトウェアレンダラを使います。')

# ---- 描画ループ ----
from PIL import Image

SILENT_MP4 = os.path.join(WORK_DIR, 'mannequin_silent.mp4')
src_reader = imageio.get_reader(SEGMENT_MP4) if SIDE_BY_SIDE else None
writer = imageio.get_writer(SILENT_MP4, fps=FPS, codec='libx264', quality=8,
                            macro_block_size=1, pixelformat='yuv420p')
for i in tqdm(range(len(render_verts)), desc='描画中'):
    extra = floor
    if backend is not None:
        img = backend.render(render_verts[i], FACES, K_RENDER, IMSHAPE, body_rgb, bg_rgb, extra)
    else:
        img = render_software(render_verts[i], FACES, K_RENDER, IMSHAPE, body_rgb, bg_rgb, extra)
    if src_reader is not None:
        try:
            src = np.asarray(Image.fromarray(src_reader.get_data(i)).resize(
                (out_w, out_h), Image.BILINEAR))
            img = np.concatenate([src, img], axis=1)
        except Exception:
            pass
    writer.append_data(img)
writer.close()
if src_reader is not None:
    src_reader.close()
print('マネキン動画（無音）:', SILENT_MP4, f'({os.path.getsize(SILENT_MP4) / 1024 ** 2:.1f} MB)')

---
## 12. 音声を合成して完成 🎬

切り出しておいた音声を重ねて `mannequin_with_audio.mp4` を書き出し、その場で再生・ダウンロードします。

In [ ]:
#@title 12. 音声付き mp4 を書き出して再生 { display-mode: "form" }
import base64
from IPython.display import HTML, display

FINAL_MP4 = os.path.join(WORK_DIR, 'mannequin_with_audio.mp4')
if AUDIO_PATH:
    run_ffmpeg(['-y', '-loglevel', 'error', '-i', SILENT_MP4, '-i', AUDIO_PATH,
                '-c:v', 'copy', '-c:a', 'aac', '-shortest', FINAL_MP4])
    print('音声を合成しました。')
else:
    run_ffmpeg(['-y', '-loglevel', 'error', '-i', SILENT_MP4, '-c', 'copy', FINAL_MP4])
    print('元動画に音声が無かったため、無音で出力しました。')

size_mb = os.path.getsize(FINAL_MP4) / 1024 ** 2
print('完成:', FINAL_MP4, f'({size_mb:.1f} MB)')

if size_mb < 60:
    b64 = base64.b64encode(open(FINAL_MP4, 'rb').read()).decode()
    display(HTML(f'<video width="480" controls loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
else:
    print('（ファイルが大きいのでインライン再生は省略しました）')

if IN_COLAB:
    from google.colab import files
    files.download(FINAL_MP4)
    files.download(MOTION_NPZ)
print('モーションデータ:', MOTION_NPZ)

---
## 13. モーションを FBX で書き出す（Cascadeur / Blender / Unity / UE 用）

抽出したモーションを **FBX（バイナリ 7.4）** でも書き出します。追加のインストールは不要です
（FBX SDK や Blender を使わず、このセルの中で直接ファイルを書きます）。

**[Cascadeur](https://cascadeur.com/) にそのまま読み込めて、リグの調整が要らない形式**にしてあります。

| Cascadeur の要件 | この書き出し |
|---|---|
| Y 軸が上・単位は cm | Y-up / cm（`UnitScaleFactor = 1`）で出力 |
| キャラクターは +Z を向き、地面に対して直立 | カメラ座標系から変換して、正面を +Z・足元を y=0 に配置 |
| **フレーム 0 がデフォルトポーズ**（T / A ポーズ・左右対称） | `FBX_DEFAULT_POSE_FRAME` でフレーム 0 に SMPL の静止姿勢（回転ゼロ・足が地面）を入れ、モーションはフレーム 1 から |
| Quick Rigging Tool が認識する骨の名前・左右の表記が統一されていること | `FBX_BONE_NAMES='unreal'` で Unreal 準拠の名前（`pelvis` / `spine_01` / `thigh_l` / `calf_r` / `ball_l` …）にそろえ、左右は `_l` / `_r` で統一 |
| メッシュが骨にスキニングされていること | SMPL のスキニングウェイトでマネキンをバインド（`FBX_INCLUDE_MESH`） |
| ルートのスケールが 1 | 原点に固定した `root` を親に置き、スケールはすべて 1 |

中身:

| 要素 | 内容 |
|---|---|
| スケルトン | `root` ＋ SMPL の 24 関節（`pelvis` がその下のルート）。骨の長さはセル 8 で固定した共通体型から計算 |
| アニメーション | 全フレームに線形キー。`pelvis` は位置＋回転、それ以外は回転のみ（XYZ オイラー角） |
| メッシュ | セル 10 のマネキンをデフォルトポーズで入れ、スケルトンにバインド（法線・マテリアル付き） |

* 動き（関節の回転・全身の位置）は `motion.npz` と同じで、8a の平滑化と 8b のリフィットも反映されます
* SMPL の「ポーズ補正ブレンドシェイプ」は FBX では表現できないため、肘や膝の曲げ部分の形はレンダリング動画とわずかに異なります（骨の動きは同一です）
* 指の骨はありません（SMPL は手を 1 関節で表すため）。Cascadeur の指スロットは空のままで構いません
* Blender で読み込んだときは、タイムラインの終了フレームが自動では変わりません。必要なら総フレーム数に合わせてください

In [ ]:
#@title 13. モーションを FBX で書き出す { display-mode: "form" }
FBX_INCLUDE_MESH = True  #@param {type:"boolean"}
FBX_GROUND_AT_ORIGIN = True  #@param {type:"boolean"}
FBX_DEFAULT_POSE_FRAME = True  #@param {type:"boolean"}
FBX_BONE_NAMES = "unreal"  #@param ["unreal", "smpl"]

import struct
import zlib

# ---- FBX 7.4 バイナリの最小ライタ（Blender の io_scene_fbx/encode_bin.py と同じ書式） ----
FBX_VERSION = 7400
FBX_KTIME = 46186158000   # 1 秒あたりの FBX 時間単位
# キーの補間指定（FBX SDK の KeyAttrFlags）。全フレームにキーを打つので線形。
FBX_KEY_LINEAR = 1 << 2 | 1 << 8 | 1 << 13 | 1 << 14
_SENTINEL = b'\0' * 13
_ALWAYS_SENTINEL = {'AnimationStack', 'AnimationLayer'}


class FbxNode:
    def __init__(self, name, *props):
        self.name, self.props, self.children = name, list(props), []

    def add(self, name, *props):
        node = FbxNode(name, *props)
        self.children.append(node)
        return node


def _fbx_prop(v):
    # (型コード, バイト列) に変換する。型は値の Python 型と numpy の dtype で決める。
    if isinstance(v, tuple) and len(v) == 2 and isinstance(v[0], str):
        kind, v = v   # 型を明示したいとき: ('L', 123) / ('d', array) など
    elif isinstance(v, (bool, np.bool_)):
        kind = 'C'
    elif isinstance(v, (int, np.integer)):
        kind = 'I' if -2 ** 31 <= int(v) < 2 ** 31 else 'L'
    elif isinstance(v, (float, np.floating)):
        kind = 'D'
    elif isinstance(v, str):
        kind = 'S'
    elif isinstance(v, bytes):
        kind = 'R'
    else:
        a = np.asarray(v)
        kind = {'f': 'd', 'i': 'i', 'u': 'i', 'b': 'b'}[a.dtype.kind]
    if kind in 'SR':
        data = v.encode('utf-8') if isinstance(v, str) else v
        return kind.encode(), struct.pack('<I', len(data)) + data
    if kind in 'CYIFDL':
        fmt = {'C': '<?', 'Y': '<h', 'I': '<i', 'F': '<f', 'D': '<d', 'L': '<q'}[kind]
        return kind.encode(), struct.pack(fmt, v)
    dtype = {'d': '<f8', 'f': '<f4', 'i': '<i4', 'l': '<i8', 'b': '?'}[kind]
    a = np.ascontiguousarray(np.asarray(v).reshape(-1), dtype=dtype)
    raw = a.tobytes()
    enc = 0 if len(raw) <= 128 else 1
    if enc:
        raw = zlib.compress(raw, 1)
    return kind.encode(), struct.pack('<3I', len(a), enc, len(raw)) + raw


def _fbx_node_bytes(node, offset, is_last):
    # ノード 1 つ分のバイト列。先頭の EndOffset はファイル先頭からの絶対位置。
    props = [_fbx_prop(p) for p in node.props]
    prop_bytes = b''.join(t + d for t, d in props)
    name = node.name.encode()
    head_len = 12 + 1 + len(name) + len(prop_bytes)
    body, pos = [], offset + head_len
    for i, child in enumerate(node.children):
        b = _fbx_node_bytes(child, pos, i == len(node.children) - 1)
        body.append(b)
        pos += len(b)
    if node.children or (not node.props and not is_last) or node.name in _ALWAYS_SENTINEL:
        body.append(_SENTINEL)
        pos += len(_SENTINEL)
    head = struct.pack('<3I', pos, len(props), len(prop_bytes)) + bytes([len(name)]) + name
    return head + prop_bytes + b''.join(body)


def write_fbx_file(path, top_nodes):
    out = bytearray(b'Kaydara FBX Binary  \x00\x1a\x00' + struct.pack('<I', FBX_VERSION))
    for i, node in enumerate(top_nodes):
        out += _fbx_node_bytes(node, len(out), i == len(top_nodes) - 1)
    out += _SENTINEL
    # フッタ（FBX SDK が期待する固定値。Blender の書き出しと同じ）
    out += b'\xfa\xbc\xab\x09\xd0\xc8\xd4\x66\xb1\x76\xfb\x83\x1c\xf7\x26\x7e' + b'\0' * 4
    pad = ((len(out) + 15) & ~15) - len(out)
    out += b'\0' * (pad or 16)
    out += struct.pack('<I', FBX_VERSION) + b'\0' * 120
    out += b'\xf8\x5a\x8c\x6a\xde\xf5\xd9\x7e\xec\xe9\x0c\xe3\x75\x8f\x29\x0b'
    with open(path, 'wb') as f:
        f.write(out)
    return path


def _p70(parent, *entries):
    # Properties70 ブロック。entries = (名前, 型, ラベル, フラグ, 値...) のタプル
    p70 = parent.add('Properties70')
    for e in entries:
        p70.add('P', *e)
    return p70


def _mat_cols(m):
    # FBX の行列は列優先（平行移動が 12〜14 番目）
    return np.asarray(m, np.float64).T.reshape(-1)


def _mesh_normals(verts, faces, smooth):
    # ByPolygonVertex 用の法線 (F*3, 3)。smooth=True なら面積重みで頂点ごとに平均する。
    v, f = np.asarray(verts, np.float64), np.asarray(faces, np.int64)
    a, b, c = v[f[:, 0]], v[f[:, 1]], v[f[:, 2]]
    fn = np.cross(b - a, c - a)   # 長さ = 面積の 2 倍 = 面積重み
    if not smooth:
        n = np.repeat(fn, 3, axis=0)
    else:
        vn = np.zeros_like(v)
        for k in range(3):
            np.add.at(vn, f[:, k], fn)
        n = vn[f.reshape(-1)]
    return n / np.maximum(np.linalg.norm(n, axis=-1, keepdims=True), 1e-12)


def build_motion_fbx(path, names, parents, rest_joints_cm, local_euler_deg, root_pos_cm, fps,
                     mesh=None, body_color=(0.85, 0.82, 0.78), take_name='Take 001',
                     root_bone_name='root', smooth_normals=True):
    # names/parents: 骨の名前と親 (J,) / rest_joints_cm: T ポーズの関節位置 (J, 3)
    # local_euler_deg: 各骨の親に対する回転 (T, J, 3)、XYZ オイラー角 [度]（行列 = Rz·Ry·Rx）
    # root_pos_cm: ルート骨の位置 (T, 3)
    # mesh: None または (頂点 (V,3) [cm, T ポーズ], 面 (F,3), スキンウェイト (V,J))
    # root_bone_name: 原点に固定する親骨の名前（None なら作らない）
    J, T = len(names), len(local_euler_deg)
    uid = iter(range(1_000_000, 10_000_000))
    ids = lambda: ('L', next(uid))
    objects, conns = FbxNode('Objects'), FbxNode('Connections')
    counts = {}

    def obj(kind, *props):
        counts[kind] = counts.get(kind, 0) + 1
        return objects.add(kind, *props)

    def connect(child, parent, prop=None):
        if prop is None:
            conns.add('C', 'OO', child, parent)
        else:
            conns.add('C', 'OP', child, parent, prop)

    def model(mid, name, kind, t, r=(0.0, 0.0, 0.0)):
        m = obj('Model', mid, name + '\x00\x01Model', kind)
        m.add('Version', 232)
        _p70(m,
             ('RotationActive', 'bool', '', '', 1),
             ('InheritType', 'enum', '', '', 1),
             ('DefaultAttributeIndex', 'int', 'Integer', '', 0),
             ('Lcl Translation', 'Lcl Translation', '', 'A', *map(float, t)),
             ('Lcl Rotation', 'Lcl Rotation', '', 'A', *map(float, r)),
             ('Lcl Scaling', 'Lcl Scaling', '', 'A', 1.0, 1.0, 1.0))
        m.add('Shading', True)
        m.add('Culling', 'CullingOff')
        return m

    def limb(name, t, r=(0.0, 0.0, 0.0)):
        attr_id, bid = ids(), ids()
        a = obj('NodeAttribute', attr_id, name + '\x00\x01NodeAttribute', 'LimbNode')
        _p70(a, ('Size', 'double', 'Number', '', 3.0))
        a.add('TypeFlags', 'Skeleton')
        model(bid, name, 'LimbNode', t, r)
        connect(attr_id, bid)
        return bid

    # ---- 骨（LimbNode） ----
    # ゲームエンジン（Unreal など）と同じく、原点に固定の 'root' を置いてその下に骨を並べる。
    root_id = limb(root_bone_name, (0.0, 0.0, 0.0)) if root_bone_name else ('L', 0)
    if root_bone_name:
        connect(root_id, ('L', 0))
    bone_ids = []
    for j in range(J):
        p = parents[j]
        if p < 0:
            t, r = root_pos_cm[0], local_euler_deg[0, j]
        else:
            t, r = rest_joints_cm[j] - rest_joints_cm[p], local_euler_deg[0, j]
        bid = limb(names[j], t, r)
        connect(bid, bone_ids[p] if p >= 0 else root_id)
        bone_ids.append(bid)

    # ---- スキン付きメッシュ（任意） ----
    if mesh is not None:
        verts, faces, weights = mesh
        mesh_id, geo_id, mat_id, skin_id, pose_id = ids(), ids(), ids(), ids(), ids()
        model(mesh_id, 'Mannequin', 'Mesh', (0.0, 0.0, 0.0))
        connect(mesh_id, ('L', 0))

        g = obj('Geometry', geo_id, 'Mannequin\x00\x01Geometry', 'Mesh')
        g.add('Vertices', ('d', np.asarray(verts, np.float64)))
        pvi = np.asarray(faces, np.int32).copy()
        pvi[:, -1] = ~pvi[:, -1]   # 多角形の最後の頂点はビット反転で区切りを表す
        g.add('PolygonVertexIndex', ('i', pvi))
        g.add('GeometryVersion', 124)
        ln = g.add('LayerElementNormal', 0)
        ln.add('Version', 102)
        ln.add('Name', '')
        ln.add('MappingInformationType', 'ByPolygonVertex')
        ln.add('ReferenceInformationType', 'Direct')
        ln.add('Normals', ('d', _mesh_normals(verts, faces, smooth_normals)))
        lm = g.add('LayerElementMaterial', 0)
        lm.add('Version', 101)
        lm.add('Name', '')
        lm.add('MappingInformationType', 'AllSame')
        lm.add('ReferenceInformationType', 'IndexToDirect')
        lm.add('Materials', ('i', np.zeros(1, np.int32)))
        layer = g.add('Layer', 0)
        layer.add('Version', 100)
        for kind in ('LayerElementNormal', 'LayerElementMaterial'):
            le = layer.add('LayerElement')
            le.add('Type', kind)
            le.add('TypedIndex', 0)
        connect(geo_id, mesh_id)

        mat = obj('Material', mat_id, 'MannequinMat\x00\x01Material', '')
        mat.add('Version', 102)
        mat.add('ShadingModel', 'Phong')
        mat.add('MultiLayer', 0)
        c = tuple(float(x) for x in body_color)
        _p70(mat,
             ('DiffuseColor', 'Color', '', 'A', *c),
             ('DiffuseFactor', 'Number', '', 'A', 1.0),
             ('SpecularFactor', 'Number', '', 'A', 0.1),
             ('Shininess', 'Number', '', 'A', 10.0))
        connect(mat_id, mesh_id)

        skin = obj('Deformer', skin_id, 'Skin\x00\x01Deformer', 'Skin')
        skin.add('Version', 101)
        skin.add('Link_DeformAcuracy', 50.0)
        connect(skin_id, geo_id)

        bind_world = []
        for j in range(J):
            m = np.eye(4)
            m[:3, 3] = rest_joints_cm[j]   # T ポーズでは全骨の回転が単位行列
            bind_world.append(m)
            idx = np.flatnonzero(weights[:, j] > 1e-4)
            cid = ids()
            cl = obj('Deformer', cid, names[j] + '\x00\x01SubDeformer', 'Cluster')
            cl.add('Version', 100)
            cl.add('UserData', '', '')
            if len(idx):
                cl.add('Indexes', ('i', idx.astype(np.int32)))
                cl.add('Weights', ('d', weights[idx, j].astype(np.float64)))
            # Transform は「骨の空間から見たメッシュ」（Blender の書き出しと同じ解釈）
            cl.add('Transform', ('d', _mat_cols(np.linalg.inv(m))))
            cl.add('TransformLink', ('d', _mat_cols(m)))
            connect(cid, skin_id)
            connect(bone_ids[j], cid)

        pose = obj('Pose', pose_id, 'BindPose\x00\x01Pose', 'BindPose')
        pose.add('Type', 'BindPose')
        pose.add('Version', 100)
        pose.add('NbPoseNodes', J + 1)
        for nid, m in [(mesh_id, np.eye(4))] + list(zip(bone_ids, bind_world)):
            pn = pose.add('PoseNode')
            pn.add('Node', nid)
            pn.add('Matrix', ('d', _mat_cols(m)))

    # ---- アニメーション ----
    ktimes = np.round(np.arange(T) * (FBX_KTIME / float(fps))).astype(np.int64)
    t_end = int(ktimes[-1])
    stack_id, layer_id = ids(), ids()
    st = obj('AnimationStack', stack_id, take_name + '\x00\x01AnimStack', '')
    _p70(st, ('LocalStop', 'KTime', 'Time', '', ('L', t_end)),
         ('ReferenceStop', 'KTime', 'Time', '', ('L', t_end)))
    obj('AnimationLayer', layer_id, 'BaseLayer\x00\x01AnimLayer', '')
    connect(layer_id, stack_id)

    def anim_channel(target, prop, values):
        values = np.asarray(values, np.float64)
        cn_id = ids()
        cn = obj('AnimationCurveNode', cn_id, prop[4].upper() + '\x00\x01AnimCurveNode', '')
        _p70(cn, *[(f'd|{ax}', 'Number', '', 'A', float(values[0, k]))
                   for k, ax in enumerate('XYZ')])
        connect(cn_id, layer_id)
        connect(cn_id, target, prop)
        for k, ax in enumerate('XYZ'):
            cv_id = ids()
            cv = obj('AnimationCurve', cv_id, '\x00\x01AnimCurve', '')
            cv.add('Default', float(values[0, k]))
            cv.add('KeyVer', 4008)
            cv.add('KeyTime', ('l', ktimes))
            cv.add('KeyValueFloat', ('f', values[:, k].astype(np.float32)))
            cv.add('KeyAttrFlags', ('i', np.array([FBX_KEY_LINEAR], np.int32)))
            cv.add('KeyAttrDataFloat', ('f', np.array([0, 0, 9.419963346924634e-30, 0], np.float32)))
            cv.add('KeyAttrRefCount', ('i', np.array([T], np.int32)))
            connect(cv_id, cn_id, f'd|{ax}')

    for j in range(J):
        if parents[j] < 0:
            anim_channel(bone_ids[j], 'Lcl Translation', root_pos_cm)
        anim_channel(bone_ids[j], 'Lcl Rotation', local_euler_deg[:, j])

    # ---- ヘッダ・グローバル設定・定義 ----
    header = FbxNode('FBXHeaderExtension')
    header.add('FBXHeaderVersion', 1003)
    header.add('FBXVersion', FBX_VERSION)
    header.add('EncryptionType', 0)
    ts = header.add('CreationTimeStamp')
    for k, v in [('Version', 1000), ('Year', 1970), ('Month', 1), ('Day', 1), ('Hour', 10),
                 ('Minute', 0), ('Second', 0), ('Millisecond', 0)]:
        ts.add(k, v)
    header.add('Creator', 'nlf mp4_to_mannequin')

    fps_modes = {120.0: 1, 100.0: 2, 60.0: 3, 50.0: 4, 48.0: 5, 30.0: 6, 25.0: 10, 24.0: 11,
                 96.0: 15, 72.0: 16}
    time_mode = next((m for f, m in fps_modes.items() if abs(f - fps) < 1e-3), 14)
    gs = FbxNode('GlobalSettings')
    gs.add('Version', 1000)
    _p70(gs,
         ('UpAxis', 'int', 'Integer', '', 1), ('UpAxisSign', 'int', 'Integer', '', 1),
         ('FrontAxis', 'int', 'Integer', '', 2), ('FrontAxisSign', 'int', 'Integer', '', 1),
         ('CoordAxis', 'int', 'Integer', '', 0), ('CoordAxisSign', 'int', 'Integer', '', 1),
         ('OriginalUpAxis', 'int', 'Integer', '', 1), ('OriginalUpAxisSign', 'int', 'Integer', '', 1),
         ('UnitScaleFactor', 'double', 'Number', '', 1.0),        # 1 単位 = 1 cm
         ('OriginalUnitScaleFactor', 'double', 'Number', '', 1.0),
         ('TimeMode', 'enum', '', '', time_mode),
         ('TimeSpanStart', 'KTime', 'Time', '', ('L', 0)),
         ('TimeSpanStop', 'KTime', 'Time', '', ('L', t_end)),
         ('CustomFrameRate', 'double', 'Number', '', float(fps)))

    docs = FbxNode('Documents')
    docs.add('Count', 1)
    doc = docs.add('Document', ids(), 'Scene', 'Scene')
    _p70(doc, ('SourceObject', 'object', '', ''),
         ('ActiveAnimStackName', 'KString', '', '', take_name))
    doc.add('RootNode', ('L', 0))

    defs = FbxNode('Definitions')
    defs.add('Version', 100)
    defs.add('Count', sum(counts.values()) + 1)
    defs.add('ObjectType', 'GlobalSettings').add('Count', 1)
    for kind, n in counts.items():
        defs.add('ObjectType', kind).add('Count', n)

    top = [header,
           FbxNode('FileId', b'\x28\xb3\x2a\xeb\xb6\x24\xcc\xc2\xbf\xc8\xb0\x2a\xa9\x2b\xfc\xf1'),
           FbxNode('CreationTime', '1970-01-01 10:00:00:000'),
           FbxNode('Creator', 'nlf mp4_to_mannequin'),
           gs, docs, FbxNode('References'), defs, objects, conns,
           FbxNode('Takes')]
    takes = top[-1]
    takes.add('Current', take_name)
    tk = takes.add('Take', take_name)
    tk.add('FileName', take_name.replace(' ', '_') + '.tak')
    tk.add('LocalTime', ('L', 0), ('L', t_end))
    tk.add('ReferenceTime', ('L', 0), ('L', t_end))
    return write_fbx_file(path, top)


# ---- SMPL のモーション -> FBX の骨アニメーション ----
# カメラ座標系（x=右 / y=下 / z=奥）を、FBX の標準である Y-up（x=右 / y=上 / z=手前）に直す。
# x 軸まわりの 180 度回転なので、全身の向き（ルートの回転）と位置にだけ掛ければよい。
CAM_TO_YUP = np.diag([1.0, -1.0, -1.0])


def smpl_forward_kinematics(pose, trans, rest_joints, parents):
    # SMPL と同じ順運動学: 関節 j の位置 = 親の位置 + 親の大域回転 × (静止姿勢での親からの差分)
    T, J = pose.shape[:2]
    R = Rotation.from_rotvec(pose.reshape(-1, 3)).as_matrix().reshape(T, J, 3, 3)
    G, P = np.empty_like(R), np.empty((T, J, 3))
    for j, p in enumerate(parents):
        if p < 0:
            G[:, j], P[:, j] = R[:, j], rest_joints[j] + trans
        else:
            G[:, j] = G[:, p] @ R[:, j]
            P[:, j] = P[:, p] + G[:, p] @ (rest_joints[j] - rest_joints[p])
    return P


def smpl_to_fbx_channels(pose, trans, rest_joints, shift):
    # 戻り値: 各骨の XYZ オイラー角 [度] (T, J, 3) と、ルート骨の位置 [cm] (T, 3)
    T, J = pose.shape[:2]
    R = Rotation.from_rotvec(pose.reshape(-1, 3)).as_matrix().reshape(T, J, 3, 3)
    R[:, 0] = CAM_TO_YUP @ R[:, 0]
    # 小文字 'xyz'（固定軸）= 行列 Rz·Ry·Rx で、FBX の回転順序 XYZ と同じ
    euler = Rotation.from_matrix(R.reshape(-1, 3, 3)).as_euler('xyz').reshape(T, J, 3)
    euler = np.rad2deg(np.unwrap(euler, axis=0))   # ±180 度の折り返しで補間が 1 回転しないように
    root = (rest_joints[0] + trans) @ CAM_TO_YUP.T + shift
    return euler, root * 100.0


# SMPL の 24 関節を Unreal / Cascadeur で標準的な骨名に対応させる（左右は _l / _r で統一）。
# Cascadeur の Quick Rigging Tool はこの命名の骨格を自動認識します。
UE_BONE_NAMES = [
    'pelvis', 'thigh_l', 'thigh_r', 'spine_01', 'calf_l', 'calf_r', 'spine_02',
    'foot_l', 'foot_r', 'spine_03', 'ball_l', 'ball_r', 'neck_01', 'clavicle_l',
    'clavicle_r', 'head', 'upperarm_l', 'upperarm_r', 'lowerarm_l', 'lowerarm_r',
    'hand_l', 'hand_r', 'middle_01_l', 'middle_01_r']

# ---- 静止姿勢（デフォルトポーズ）の関節・頂点を体モデルから作る ----
fbx_repose_fn, _ = get_repose_fn()
assert fbx_repose_fn is not None, '体モデルが使えないため FBX を書き出せません。'
betas_fbx = motion['betas'][0].astype(np.float32)   # セル 8 / 8b で全フレーム共通にした体型
v_rest, j_rest = repose(np.zeros((1, N_JOINTS, 3), np.float32), betas_fbx[None],
                        np.zeros((1, 3), np.float32), fbx_repose_fn)
v_rest, j_rest = v_rest[0] / 1000.0, j_rest[0] / 1000.0   # [m]

pose_fbx = motion['pose'].astype(np.float64)
trans_fbx = motion['trans'].astype(np.float64)
fk_err = np.abs(smpl_forward_kinematics(pose_fbx, trans_fbx, j_rest, SMPL_PARENTS) * 1000.0
                - motion['joints3d']).max()
print(f'骨アニメーションの自己検証: motion.npz の関節との最大差 {fk_err:.2f} mm')
if fk_err > 20.0:
    print('⚠️ 差が大きめです（再ポーズが使えなかった場合に起こります）。')

# ---- 置き場所: 床を y=0 に、水平方向は全身の平均位置を原点に ----
shift = np.zeros(3)
if FBX_GROUND_AT_ORIGIN:
    floor_mm = float(np.percentile(motion['vertices3d'][..., 1], 99.7))
    shift[1] = floor_mm / 1000.0   # y を反転するので、カメラ座標の床の高さ F は -F になる
    root_yup = (j_rest[0] + trans_fbx) @ CAM_TO_YUP.T
    shift[[0, 2]] = -root_yup[:, [0, 2]].mean(0)
euler_fbx, root_cm = smpl_to_fbx_channels(pose_fbx, trans_fbx, j_rest, shift)

# ---- マネキンのメッシュ（セル 10 と同じ形）とスキンウェイト ----
fbx_mesh = None
if FBX_INCLUDE_MESH and 'FACES' in globals():
    lbs = try_get_lbs_weights(nlf_model, BODY_MODEL_NAME, len(v_rest))
    if lbs is None and BODY_MODEL is not None:
        lbs = BODY_MODEL.weights.detach().float().cpu().numpy()
    if lbs is None:
        # スキンウェイトが取れなければ、各頂点を最も近い関節に 100% 割り当てる
        near = np.linalg.norm(v_rest[:, None] - j_rest[None], axis=-1).argmin(1)
        lbs = np.eye(N_JOINTS, dtype=np.float32)[near]
    w = lbs[VERTEX_MAP]
    w = w / np.maximum(w.sum(1, keepdims=True), 1e-8)
    fbx_mesh = (v_rest[VERTEX_MAP] * 100.0, FACES, w)
elif FBX_INCLUDE_MESH:
    print('⚠️ セル 10 のマネキンがまだ無いので、骨だけの FBX にします。')

# ---- フレーム 0 にデフォルトポーズ（回転ゼロ・足を地面に）を 1 枚入れる ----
# Cascadeur など「フレーム 0 が T/A ポーズであること」を前提にするツール向け。
# 姿勢は SMPL の静止姿勢そのもの（左右対称・直立・正面が +Z）なので、
# 読み込んだ側でリグを組み直さなくてもそのまま使えます。
if FBX_DEFAULT_POSE_FRAME:
    ground = (fbx_mesh[0][:, 1].min() / 100.0 if fbx_mesh is not None else j_rest[:, 1].min())
    rest_root = (j_rest[0] - [0.0, ground, 0.0]) * 100.0
    euler_fbx = np.concatenate([np.zeros((1, N_JOINTS, 3)), euler_fbx])
    root_cm = np.concatenate([rest_root[None], root_cm])

MOTION_FBX = os.path.join(WORK_DIR, 'motion.fbx')
build_motion_fbx(MOTION_FBX,
                 UE_BONE_NAMES if FBX_BONE_NAMES == 'unreal' else SMPL_JOINT_NAMES,
                 SMPL_PARENTS, j_rest * 100.0, euler_fbx, root_cm, FPS, mesh=fbx_mesh,
                 body_color=hex2rgb(BODY_COLOR) if 'BODY_COLOR' in globals() else (0.85, 0.82, 0.78),
                 smooth_normals=bool(globals().get('use_smpl_mesh', False)))
print('FBX を書き出しました:', MOTION_FBX, f'({os.path.getsize(MOTION_FBX) / 1024 ** 2:.1f} MB)')
print(f'  骨 {N_JOINTS} 本（root + {FBX_BONE_NAMES} 命名） / {len(euler_fbx)} フレーム @ {FPS:g} fps / '
      + (f'スキン付きメッシュ（頂点 {len(fbx_mesh[0])} / 面 {len(fbx_mesh[1])}）'
         if fbx_mesh is not None else 'メッシュなし'))
if FBX_DEFAULT_POSE_FRAME:
    print('  フレーム 0 = デフォルトポーズ、フレーム 1 以降がモーションです。')

if IN_COLAB:
    from google.colab import files
    files.download(MOTION_FBX)

---
## 14. MMD 用の VMD モーションを書き出す（nlf2vmd）

抽出したモーションを **MikuMikuDance / MikuMikuMoving 用の VMD**（`motion.vmd`）に変換します。
変換はこのリポジトリの [`nlf2vmd`](nlf2vmd/README.md) パッケージで行い、処理の仕様は [`vmd.md`](vmd.md) のとおりです。
（Colab で実行したときは、このセルが GitHub から `nlf2vmd` を取得します）

変換の流れ（順番は固定）:

1. 読み込み・正規化（カメラ座標 → Y 上向き、体型を中央値 1 つに固定、30fps へリサンプル）
2. 姿勢のジッター制御（クォータニオンの符号揃え → 単独フレームの外れ値除去 → 関節グループ別の One Euro フィルタ）
3. FK で関節位置と、**かかと・つま先**の位置を計算
4. 床面の推定（低速のかかと・つま先に RANSAC で平面を当てはめ、床を y=0 に。補正は定数だけ）
5. MMD スケールへ変換（モデルの脚長 ÷ SMPL の脚長）
6. 接地判定（高さと水平速度、ヒステリシス付き）
7. 足ＩＫ（接地中は位置と回転をロック → 遊脚を平滑化 → 境界をなめらかにつなぐ → 床下を 0 にクランプ）
8. センターの安定化（軸別の平滑化、または接地足からの逆算）＋脚が**届く高さへセンターを下げる**
9. 上半身・腕の回転を、T ポーズ（SMPL）と A ポーズ（MMD）の違いを補正してリターゲット
10. VMD の書き出しと、処理前後の評価指標（JSON）・グラフ（PNG）の出力

| 設定 | 意味 |
|---|---|
| `PMX_PATH` | 使うモデルの `.pmx`（脚長とボーンの初期位置を読みます）。空なら標準的な体格の寸法で変換します。**実際に使うモデルを指定するのがおすすめ**です |
| `UPLOAD_PMX` | Colab で `.pmx` をアップロードして使う（`PMX_PATH` より優先） |
| `VMD_INPUT` | `unsmoothed`（既定）= セル 8a の MMPose 平滑化の前のモーション。平滑化は nlf2vmd 側で関節ごとに掛けます / `smoothed` = 8a・8b の後 |
| `CENTER_MODE` | `A` = センターを軸別に平滑化（既定）/ `B` = 接地している足から骨盤の位置を逆算（足とセンターが一体に動く） |
| `DEPTH_SCALE` | 奥行き方向の移動量だけを縮める倍率（単眼推定は奥行きがぶれやすいので、前後にふらつくときは 0.5 など） |
| `FLIP_FACING` | MMD で体が後ろを向く・進行方向が逆になるときに `True` |
| `THIN_KEYS` | キーを間引く（接地の開始・終了の足ＩＫキーは必ず残します） |
| `VMD_CONFIG_PATH` | 細かい設定を書いた YAML（[`nlf2vmd/default_config.yaml`](nlf2vmd/default_config.yaml) をコピーして、変えたい項目だけ書く） |

**MMD での確認（手動）**: `motion.vmd` をモデルに読み込み、足の滑り・埋まり・膝の暴れ・全身の震えを確認してください。
問題があれば、フレーム番号と症状をメモし、下のグラフ（`vmd_diag/`）と照らし合わせて設定を調整します。
`motion_for_vmd.npz` と `smpl_body_model.npz` をダウンロードしておけば、手元の PC でも
`python -m nlf2vmd motion_for_vmd.npz --pmx モデル.pmx --set center.mode=B` のように設定を変えて作り直せます（GPU 不要）。

In [ ]:
#@title 14. VMD（MMD 用モーション）を書き出す { display-mode: "form" }
PMX_PATH = ''  #@param {type:"string"}
UPLOAD_PMX = False  #@param {type:"boolean"}
VMD_INPUT = "unsmoothed"  #@param ["unsmoothed", "smoothed"]
CENTER_MODE = "A"  #@param ["A", "B"]
DEPTH_SCALE = 1.0  #@param {type:"number"}
FLIP_FACING = False  #@param {type:"boolean"}
THIN_KEYS = False  #@param {type:"boolean"}
VMD_CONFIG_PATH = ''  #@param {type:"string"}
NLF2VMD_REPO = 'https://github.com/yamak493/nlf'  #@param {type:"string"}
NLF2VMD_BRANCH = 'main'  #@param {type:"string"}


def import_nlf2vmd():
    # リポジトリの中で実行していればそのまま import、Colab なら GitHub から取得する
    try:
        import nlf2vmd
        return nlf2vmd
    except ImportError:
        pass
    src_dir = os.path.join(WORK_DIR, 'nlf_src')
    if not os.path.isdir(os.path.join(src_dir, 'nlf2vmd')):
        print('nlf2vmd を取得中:', NLF2VMD_REPO, f'({NLF2VMD_BRANCH})')
        shutil.rmtree(src_dir, ignore_errors=True)
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', NLF2VMD_BRANCH,
                        NLF2VMD_REPO, src_dir], check=True)
    sys.path.insert(0, src_dir)
    import nlf2vmd
    return nlf2vmd


nlf2vmd = import_nlf2vmd()
from nlf2vmd.body_model import BodyModel
from nlf2vmd.diagnostics import format_metrics

# ---- 対象モデル（PMX） ----
if UPLOAD_PMX and IN_COLAB:
    from google.colab import files
    print('.pmx ファイルを選んでください（ボーン情報だけを使うので、テクスチャは不要です）')
    for name, data in files.upload().items():
        if name.lower().endswith('.pmx'):
            PMX_PATH = os.path.join(WORK_DIR, os.path.basename(name))
            with open(PMX_PATH, 'wb') as f:
                f.write(data)
if PMX_PATH:
    assert os.path.exists(PMX_PATH), f'PMX が見つかりません: {PMX_PATH}'
    print('対象モデル:', PMX_PATH)

# ---- SMPL の体モデルを NLF の TorchScript から書き出す（公式ファイル不要） ----
SMPL_BODY_NPZ = os.path.join(WORK_DIR, 'smpl_body_model.npz')
BodyModel.from_torch(getattr(nlf_model.body_models, BODY_MODEL_NAME)).save_npz(SMPL_BODY_NPZ)

# ---- 変換の入力（カメラ座標のまま渡す。Y 上向きへの変換は nlf2vmd が行う） ----
src_motion = MOTION_UNSMOOTHED if VMD_INPUT == 'unsmoothed' else motion
VMD_INPUT_NPZ = os.path.join(WORK_DIR, 'motion_for_vmd.npz')
np.savez_compressed(
    VMD_INPUT_NPZ, pose=src_motion['pose'], betas=src_motion['betas'], trans=src_motion['trans'],
    joints3d=src_motion['joints3d'], valid=valid, fps=np.float32(FPS),
    coord_system=np.array('camera'))

overrides = [f'center.mode={CENTER_MODE}', f'scale.depth_scale={float(DEPTH_SCALE)}',
             f'floor.flip_facing={str(bool(FLIP_FACING)).lower()}',
             f'vmd.thin_keys={str(bool(THIN_KEYS)).lower()}']
MOTION_VMD = os.path.join(WORK_DIR, 'motion.vmd')
VMD_DIAG_DIR = os.path.join(WORK_DIR, 'vmd_diag')
vmd_result = nlf2vmd.convert(VMD_INPUT_NPZ, MOTION_VMD, pmx=PMX_PATH or None,
                             body_model=SMPL_BODY_NPZ, config=VMD_CONFIG_PATH or None,
                             overrides=overrides, diag_dir=VMD_DIAG_DIR)

print('\n評価指標（処理前 → 処理後）:')
for line in format_metrics(vmd_result.metrics):
    print('  ' + line)
print('詳細:', vmd_result.diagnostics_path)

# ---- グラフ: 接地判定・センター・届く高さへの補正量・足ＩＫの上面図 ----
from IPython.display import Image as IPImage
for path in vmd_result.plot_paths.values():
    display(IPImage(filename=path))

if IN_COLAB:
    from google.colab import files
    files.download(MOTION_VMD)

---
## 15. うまくいかないときは

| 症状 | 対処 |
|---|---|
| `CUDA out of memory` | `BATCH_SIZE` を 1〜2 に、`MAX_HEIGHT` を 480 に下げる |
| GPU が無いというエラー | Colab のランタイムを GPU に変更して最初から実行し直す |
| 人物が検出されない | 区間を人物が大きく写っているところに変える。セル 7 の `DETECTOR_THRESHOLD` を 0.15 程度に下げる |
| 別の人に乗り移る | `PERSON_SELECT` を変える。セル 7 の追跡しきい値 `1.5`（m）を小さくする |
| 体が細かく震える | セル 8a の `SMOOTH_FILTER` を `smoothnet_t8` → `smoothnet_t16` の順に強くする。`NUM_AUG` を 5 に上げる（推定ノイズを発生源で減らす） |
| 動きが鈍い・キメのポーズが浅くなる | セル 8a の `SMOOTH_FILTER` を `savizky_golay`（既定）に戻すか、`none` にする |
| GPU が余っている | `BATCH_SIZE` → `NUM_AUG` の順に上げる（クロップ枚数 = フレーム数 × 人数 × NUM_AUG） |
| マネキンが小さい / 見切れる | `CAMERA_MODE='fit'` にする。`fit_camera` の `margin` を調整する |
| 全身が入っていない動画 | NLF は部分的な人体でも推定しますが、全身が写っている区間のほうが安定します |
| FBX を読み込むとマネキンが寝ている / 小さい | 読み込み側の軸設定を「Y-up」、単位を「cm」にする（Blender・Cascadeur は既定のままで OK） |
| Cascadeur の Quick Rigging Tool が骨格を認識しない | `FBX_BONE_NAMES='unreal'`（既定）で書き出す。ミラー設定は `_l` / `_r` を指定する |
| 処理が遅い | `TARGET_FPS` を 15 に、`MAX_HEIGHT` を 480 に下げる |
| MMD で体が後ろを向く / 進行方向が逆 | セル 14 の `FLIP_FACING=True` |
| MMD で腕や脚の向き・長さが合わない | セル 14 の `PMX_PATH` に実際に使うモデルを指定する（脚長と A ポーズの角度をモデルから読みます） |
| MMD で足が滑る / 接地のタイミングが合わない | `vmd_diag/contact.png` で接地判定（緑の帯）を確認し、設定の `contact.*`（高さ・速度のしきい値）を調整する |
| MMD でセンターが前後にふらつく | セル 14 の `DEPTH_SCALE` を 0.5 などに下げる。または `CENTER_MODE="B"` |
| MMD で膝が伸び切る・暴れる | 設定の `center.reach_ratio` を 0.95 などに下げる（センターが少し下がります） |
| MMD で全身が震える | 設定の `jitter.one_euro.groups.*.min_cutoff` を小さくする（強く平滑化）。`NUM_AUG` を上げる |

### 出力ファイル

| ファイル | 内容 |
|---|---|
| `nlf_mannequin/motion.npz` | 抽出したモーションデータ（`pose` (T,24,3) 回転ベクトル、`betas`、`trans`、`joints3d`、`vertices3d`、`fps` など）。セル 8 / 8a / 8b のどれを実行しても最新の状態で保存し直されます（掛けた平滑化フィルタは `smooth_filter` に記録） |
| `nlf_mannequin/mannequin_silent.mp4` | マネキン動画（無音） |
| `nlf_mannequin/mannequin_with_audio.mp4` | **最終出力**（音声付き） |
| `nlf_mannequin/motion.vmd` | MMD 用のモーション（センター・グルーブ・足ＩＫ・上半身と腕の回転。全フレームにキー） |
| `nlf_mannequin/vmd_diag/` | VMD 変換の診断出力（`diagnostics.json`: 処理前後の評価指標と接地区間、PNG: 判定の確認用グラフ） |
| `nlf_mannequin/motion_for_vmd.npz` / `smpl_body_model.npz` | VMD 変換の入力と SMPL 体モデル。手元で `python -m nlf2vmd` を使って作り直すときに使います |
| `nlf_mannequin/motion.fbx` | モーションの FBX（`root` ＋ SMPL 24 関節のスケルトン＋スキン付きマネキン＋アニメーション。Y-up / cm。Cascadeur にそのまま読み込めます） |

### 参考

* NLF: <https://github.com/isarandi/nlf> — [NeurIPS 2024 論文](https://arxiv.org/abs/2407.07532)
* NLF v0.3.2 リリース: <https://github.com/isarandi/nlf/releases/tag/v0.3.2>
* SMPLFitter: <https://github.com/isarandi/smplfitter>
* MMPose（0.x、Smoother / 時間フィルタ）: <https://github.com/open-mmlab/mmpose/tree/0.x>
* SmoothNet: <https://github.com/cure-lab/SmoothNet> — [論文](https://arxiv.org/abs/2112.13715)

```
@article{sarandi2024nlf,
    title   = {Neural Localizer Fields for Continuous 3D Human Pose and Shape Estimation},
    author  = {S\'ar\'andi, Istv\'an and Pons-Moll, Gerard},
    journal = {Advances in Neural Information Processing Systems (NeurIPS)},
    year    = {2024}
}
```